# Visualization of the 3d Perron tree construnction - Step by step
The aim of the following notebook is to illustrate in 3 dimensions most of the lemmas or proofs we carry out in the paper, for a more intuitive understanding of them and their higher-dimensional counterparts.


## Movement of corner pieces $\Sigma_i$ radially inwards towards the barycenter $t\mathbf{u}_i=t(\mathbf{o}-\mathbf{x}_i)$


d=3

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 1. Setup Parameters for R_{4,4}
# ==========================================
d = 4
c = 0.75  # (d-1)/d

# Vertices of Sigma (R_{4,4})
vertices = [
    np.array([4, 4, 4]), # x1
    np.array([0, 4, 4]), # x2
    np.array([0, 0, 4]), # x3
    np.array([0, 0, 0])  # x4
]
o = np.mean(vertices, axis=0) # Barycenter (1, 2, 3)

corner_colors = ["#EF553B", "#00CC96", "#636EFA", "#AB63FA"]
tetra_edges = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

# ==========================================
# 2. Geometry Helper Functions
# ==========================================

def get_wireframe_trace(pts, color, name, width=4):
    x, y, z = [], [], []
    for i, j in tetra_edges:
        x.extend([pts[i][0], pts[j][0], None])
        y.extend([pts[i][1], pts[j][1], None])
        z.extend([pts[i][2], pts[j][2], None])
    return go.Scatter3d(
        x=x, y=y, z=z, mode='lines',
        line=dict(color=color, width=width),
        name=name, legendgroup=name
    )

def get_fill_trace(pts, color, name, opacity=0.15):
    return go.Mesh3d(
        x=[p[0] for p in pts], y=[p[1] for p in pts], z=[p[2] for p in pts],
        i=[0, 0, 0, 1], j=[1, 1, 2, 2], k=[2, 3, 3, 3],
        opacity=opacity, color=color, name=name, legendgroup=name, showlegend=False
    )

# ==========================================
# 3. Build Animation
# ==========================================
fig = go.Figure()
steps = []
t_values = np.linspace(0, 1 - c, 25)

for t in t_values:
    frame_data = []

    # Fixed Point: Barycenter
    frame_data.append(go.Scatter3d(
        x=[o[0]], y=[o[1]], z=[o[2]],
        mode='markers', marker=dict(size=8, color='black'),
        name='Barycenter (o)'
    ))

    # Parent Simplex
    frame_data.append(get_wireframe_trace(vertices, "rgba(150,150,150,0.2)", "Sigma", width=1))

    for i, xi in enumerate(vertices):
        # Calculate vertices for R_i(t)
        ri_verts = [c * v + (1 - c - t) * xi + t * o for v in vertices]

        # The extreme corner of R_i(t)
        extreme_corner = c * xi + (1 - c - t) * xi + t * o

        # 1. Corner Piece Fill & Wireframe
        frame_data.append(get_fill_trace(ri_verts, corner_colors[i], f"R_{i+1}"))
        frame_data.append(get_wireframe_trace(ri_verts, corner_colors[i], f"R_{i+1}"))

        # 2. Vector Line (Thick)
        frame_data.append(go.Scatter3d(
            x=[extreme_corner[0], o[0]], y=[extreme_corner[1], o[1]], z=[extreme_corner[2], o[2]],
            mode='lines', line=dict(color=corner_colors[i], width=6),
            showlegend=False, hoverinfo='skip'
        ))

        # 3. Arrowhead using a Cone (points towards o)
        # Vector direction is o - extreme_corner
        direction = o - extreme_corner
        # We place the cone slightly before 'o' so it's visible
        frame_data.append(go.Cone(
            x=[o[0]], y=[o[1]], z=[o[2]],
            u=[direction[0]], v=[direction[1]], w=[direction[2]],
            sizemode="absolute", sizeref=0.3,
            colorscale=[[0, corner_colors[i]], [1, corner_colors[i]]],
            showscale=False, name=f"Arrow {i+1}", hoverinfo='skip'
        ))

    if t == t_values[0]:
        for trace in frame_data: fig.add_trace(trace)

    steps.append({
        "method": "restyle",
        "args": [{"x": [getattr(d, 'x', None) for d in frame_data],
                  "y": [getattr(d, 'y', None) for d in frame_data],
                  "z": [getattr(d, 'z', None) for d in frame_data],
                  "u": [getattr(d, 'u', None) for d in frame_data],
                  "v": [getattr(d, 'v', None) for d in frame_data],
                  "w": [getattr(d, 'w', None) for d in frame_data]}],
        "label": f"{t:.2f}"
    })

# ==========================================
# 4. Clean Layout
# ==========================================
fig.update_layout(
    template="none",
    showlegend=False,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='white',
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "t: "},
        "pad": {"t": 50},
        "steps": steps
    }]
)

fig.show()

d=2

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 1. Setup Parameters for R_{3,3}
# ==========================================
d = 3
c = 2/3  # (d-1)/d

# Vertices of Sigma (R_{3,3})
x1 = np.array([3, 3])
x2 = np.array([0, 3])
x3 = np.array([0, 0])
vertices = [x1, x2, x3]

# Barycenter o = (1, 2)
o = np.mean(vertices, axis=0)

corner_colors = ["#EF553B", "#00CC96", "#636EFA"]
tri_edges = [(0, 1), (1, 2), (2, 0)]

# ==========================================
# 2. Geometry Helper Functions
# ==========================================

def get_outline_trace(pts, color, name, width=4):
    x, y = [], []
    for i, j in tri_edges:
        x.extend([pts[i][0], pts[j][0], None])
        y.extend([pts[i][1], pts[j][1], None])
    return go.Scatter(
        x=x, y=y, mode='lines',
        line=dict(color=color, width=width),
        name=name, legendgroup=name
    )

def get_fill_trace(pts, color, name, opacity=0.2):
    x = [p[0] for p in pts] + [pts[0][0]]
    y = [p[1] for p in pts] + [pts[0][1]]
    return go.Scatter(
        x=x, y=y, fill="toself",
        mode='lines', line=dict(width=0),
        fillcolor=color, opacity=opacity,
        name=name, legendgroup=name, showlegend=False
    )

# ==========================================
# 3. Build Animation
# ==========================================
fig = go.Figure()
steps = []
t_values = np.linspace(0, 1 - c, 30)

for t in t_values:
    frame_data = []
    current_annotations = []

    # 1. Barycenter
    frame_data.append(go.Scatter(
        x=[o[0]], y=[o[1]],
        mode='markers', marker=dict(size=12, color='black'),
        name='Barycenter (o)'
    ))

    # 2. Parent Simplex Sigma
    frame_data.append(get_outline_trace(vertices, "rgba(150,150,150,0.3)", "Sigma", width=2))

    for i, xi in enumerate(vertices):
        # R_i(t) vertices
        ri_verts = [c * v + (1 - c - t) * xi + t * o for v in vertices]
        extreme_corner = c * xi + (1 - c - t) * xi + t * o

        # Add Geometry
        frame_data.append(get_fill_trace(ri_verts, corner_colors[i], f"R_{i+1}"))
        frame_data.append(get_outline_trace(ri_verts, corner_colors[i], f"R_{i+1}"))

        # Vector Line
        frame_data.append(go.Scatter(
            x=[extreme_corner[0], o[0]], y=[extreme_corner[1], o[1]],
            mode='lines', line=dict(color=corner_colors[i], width=4),
            showlegend=False, hoverinfo='skip'
        ))

        # Arrows (Pointing to o)
        current_annotations.append(dict(
            x=o[0], y=o[1],
            ax=extreme_corner[0], ay=extreme_corner[1],
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=2,
            arrowcolor=corner_colors[i]
        ))

    if t == t_values[0]:
        for trace in frame_data: fig.add_trace(trace)
        fig.update_layout(annotations=current_annotations)

    steps.append({
        "method": "update",
        "args": [
            {"x": [getattr(d, 'x', None) for d in frame_data],
             "y": [getattr(d, 'y', None) for d in frame_data]},
            {"annotations": current_annotations}
        ],
        "label": f"{t:.2f}"
    })

# ==========================================
# 4. Corrected Layout
# ==========================================
fig.update_layout(
    showlegend=False,
    template="plotly_white",
    xaxis=dict(visible=False, range=[-0.5, 3.5]),
    yaxis=dict(visible=False, range=[-0.5, 3.5], scaleanchor="x", scaleratio=1),
    plot_bgcolor='white',  # Use plot_bgcolor instead of bgcolor
    paper_bgcolor='white',
    margin=dict(l=20, r=20, b=20, t=60),
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "t: "},
        "pad": {"t": 50},
        "steps": steps
    }]
)

fig.show()

## Height at which the $T_i$ corner pieces exit the core $C(t)$

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np

# Original 3D Geometry
v_apex = np.array([1, 2, 1])
# Base vertices of Sigma (R3,3)
v1, v2, v3 = np.array([3, 3, 0]), np.array([0, 3, 0]), np.array([0, 0, 0])
o = np.array([1, 2, 0]) # Base barycenter

# Corner cells of the base (R3,2 equivalents inside R3,3)
# These are the triangles at the corners of the base with side length 2
base_corners = {
    1: np.array([[3, 3, 0], [1, 3, 0], [1, 1, 0]]), # v1 + R3,2
    2: np.array([[0, 3, 0], [2, 3, 0], [0, 1, 0]]), # v2 + R3,2
    3: np.array([[0, 0, 0], [0, 2, 0], [2, 2, 0]])  # v3 + R3,2
}

colors = {1: 'red', 2: 'blue', 3: 'green'}
t_values = np.linspace(-1/3, 0, 31)
frames = []

for t in t_values:
    frame_data = []

    # Radial displacement vectors u_i
    u = {
        1: np.array([2, 1, 0]),
        2: np.array([-1, 1, 0]),
        3: np.array([-1, -2, 0])
    }

    # 1. Core C(t): Homothety of parent simplex (Base Sigma + Apex v_apex)
    parent_verts = np.vstack([[v1, v2, v3], v_apex])
    core_verts = o + (1 + t) * (parent_verts - o)
    edges = [0, 1, 2, 0, 3, 1, 3, 2]
    frame_data.append(go.Scatter3d(
        x=core_verts[edges, 0], y=core_verts[edges, 1], z=core_verts[edges, 2],
        mode='lines', line=dict(color='black', width=3),
        showlegend=False
    ))

    # Hyperplane height for intersection
    h_height = 1 + 3 * t

    # 2. Plot S_i(t) = conv(v_i + R_{3,2} + t*u_i, v + t*u_i)
    for i in range(1, 4):
        # Shifted base of the corner piece
        si_base = base_corners[i] + t * u[i]
        # Shifted apex for this specific corner piece
        si_apex = v_apex + t * u[i]

        # S_i(t) vertices
        si_verts = np.vstack([si_base, si_apex])
        x, y, z = si_verts[:, 0], si_verts[:, 1], si_verts[:, 2]

        # Plot Volume (Tetrahedron)
        frame_data.append(go.Mesh3d(
            x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
            color=colors[i], opacity=0.4, flatshading=True
        ))

        # Plot Wireframe
        frame_data.append(go.Scatter3d(
            x=x[edges], y=y[edges], z=z[edges],
            mode='lines', line=dict(color=colors[i], width=2), showlegend=False
        ))

        # 3. Intersection of S_i(t) with height z = 1 + 3t
        # Since the cone goes from z=0 to z=1, the slice at h is a scaled base
        s_factor = 1.0 - h_height

        if 0 <= s_factor <= 1:
            # Intersection vertices: scale si_base towards si_apex
            # Vertices move from si_base (s=1) to si_apex (s=0)
            intersect_verts = si_apex + s_factor * (si_base - si_apex)
            intersect_verts[:, 2] = h_height # Force precision

            # Yellow fill
            frame_data.append(go.Mesh3d(
                x=intersect_verts[:, 0], y=intersect_verts[:, 1], z=intersect_verts[:, 2],
                i=[0], j=[1], k=[2],
                color="yellow", opacity=0.8, showlegend=False
            ))

            # Orange border
            line_pts = intersect_verts[[0,1,2,0], :]
            frame_data.append(go.Scatter3d(
                x=line_pts[:,0], y=line_pts[:,1], z=line_pts[:,2],
                mode='lines', line=dict(color="orange", width=5), showlegend=False
            ))

    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-1, 4], visible=False),
        yaxis=dict(range=[-1, 4], visible=False),
        zaxis=dict(range=[0, 2], visible=False),
        aspectmode='cube', bgcolor='white'
    ),
    updatemenus=[{
        "buttons": [{"args": [None, {"frame": {"duration": 100, "redraw": True}}],
                     "label": "Play", "method": "animate"}],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],
        "x": 0.1, "len": 0.9
    }],
    width=900, height=800, showlegend=False
)

fig.show()

# Minimal index containment criterion for the regular simplicial subdivision of $R_{d,d}$

d=4

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np
from itertools import permutations

def is_in_R43(x):
    eps = 1e-9
    return (-eps <= x[0] <= x[1] + eps and x[1] <= x[2] + eps and x[2] <= 3 + eps)

def get_minimal_index(vertices):
    vs = {1: [1, 1, 1], 2: [0, 1, 1], 3: [0, 0, 1], 4: [0, 0, 0]}
    for i in range(1, 5):
        v_i = np.array(vs[i])
        if all(is_in_R43(v - v_i) for v in vertices):
            return i
    return 4

# 1. Generate the base cells
cells_data = []
for y1 in range(4):
    for y2 in range(y1, 4):
        for y3 in range(y2, 4):
            base = np.array([y1, y2, y3], dtype=float)
            for p in permutations([0, 1, 2]):
                verts = [base.copy()]
                curr = base.copy()
                for axis in p:
                    curr[axis] += 1
                    verts.append(curr.copy())
                if all(0 <= v[0] <= v[1]+1e-7 and v[1] <= v[2]+1e-7 and v[2] <= 4+1e-7 for v in verts):
                    cells_data.append((np.array(verts), get_minimal_index(verts)))

# Define the major vertices of the parent simplex R4,4
v_ext = {
    1: np.array([4, 4, 4]),
    2: np.array([0, 4, 4]),
    3: np.array([0, 0, 4]),
    4: np.array([0, 0, 0])
}

# Define the boundary vertices for each Si piece (The R4,3 blocks)
# These represent the corners of the "large" tetrahedra
Si_corners = {
    1: np.array([[4,4,4], [1,4,4], [1,1,4], [1,1,1]]),
    2: np.array([[0,4,4], [3,4,4], [0,1,4], [0,1,1]]),
    3: np.array([[0,0,4], [0,3,4], [3,3,4], [0,0,1]]),
    4: np.array([[0,0,0], [0,0,3], [0,3,3], [3,3,3]])
}

colors = {1: 'red', 2: 'blue', 3: 'green', 4: 'orange'}
t_values = np.linspace(-0.25, 0.25, 23)
frames = []

for t in t_values:
    frame_data = []
    displacements = {
        1: np.array([3*t, 2*t, 1*t]),
        2: np.array([-1*t, 2*t, 1*t]),
        3: np.array([-1*t, -2*t, 1*t]),
        4: np.array([-1*t, -2*t, -3*t])
    }

    # 1. Add the Coloured Regions (Sub-cells with no outlines)
    for cell, idx in cells_data:
        shifted = cell + displacements[idx]
        x, y, z = shifted[:, 0], shifted[:, 1], shifted[:, 2]
        frame_data.append(go.Mesh3d(
            x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
            color=colors[idx], opacity=0.4, flatshading=True,
            hoverinfo='skip', showlegend=False
        ))

    # 2. Add only the OUTER wireframe for each colored region
    edges = [0,1,2,0,3,1,3,2] # Tetrahedron edge sequence
    for i in range(1, 5):
        outer_v = Si_corners[i] + displacements[i]
        frame_data.append(go.Scatter3d(
            x=outer_v[edges, 0], y=outer_v[edges, 1], z=outer_v[edges, 2],
            mode='lines',
            line=dict(color='black', width=3),
            showlegend=False
        ))

    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-2, 6], visible=False),
        yaxis=dict(range=[-2, 6], visible=False),
        zaxis=dict(range=[-2, 6], visible=False),
        aspectmode='cube', bgcolor='white'
    ),
    updatemenus=[{
        "buttons": [{"args": [None, {"frame": {"duration": 200, "redraw": True}}],
                     "label": "Play", "method": "animate"}],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],
        "x": 0.1, "len": 0.9
    }],
    width=900, height=800, showlegend=False
)

fig.show()

d=3

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np

def is_in_R32(x):
    eps = 1e-9
    return (-eps <= x[0] <= x[1] + eps and x[1] <= 2 + eps)

def get_minimal_index(vertices):
    vs = {1: np.array([1, 1]), 2: np.array([0, 1]), 3: np.array([0, 0])}
    for i in range(1, 4):
        if all(is_in_R32(v - vs[i]) for v in vertices):
            return i
    return 3

# 1. Generate the 2D sub-cells for coloring
cells_data = []
for y1 in range(3):
    for y2 in range(y1, 3):
        t1 = np.array([(y1, y2), (y1+1, y2+1), (y1, y2+1)], dtype=float)
        cells_data.append((t1, get_minimal_index(t1)))
        if y1 + 1 <= y2:
            t2 = np.array([(y1, y2), (y1+1, y2), (y1+1, y2+1)], dtype=float)
            cells_data.append((t2, get_minimal_index(t2)))

# 2. Define the outlines of the full regions Ri (2x2 corner triangles)
Ri_outlines = {
    1: np.array([[3, 3], [1, 3], [1, 1]]), # Full Red R1
    2: np.array([[0, 3], [2, 3], [0, 1]]), # Full Blue R2
    3: np.array([[0, 0], [0, 2], [2, 2]])  # Full Green R3
}

colors = {1: 'red', 2: 'blue', 3: 'green'}
t_values = np.linspace(-1/3, 1/3, 21)
frames = []

for t in t_values:
    frame_data = []
    displacements = {
        1: np.array([2*t, 1*t]),
        2: np.array([-1*t, 1*t]),
        3: np.array([-1*t, -2*t])
    }

    # Add Coloured Regions WITHOUT any lines
    for cell, idx in cells_data:
        shifted = cell + displacements[idx]
        x, y = shifted[:, 0], shifted[:, 1]
        frame_data.append(go.Scatter(
            x=np.append(x, x[0]), y=np.append(y, y[0]),
            fill="toself",
            line=dict(width=0), # No lines at all on subcells
            fillcolor=colors[idx], opacity=0.5, mode='lines',
            hoverinfo='skip'
        ))

    # Add the OUTLINES of the full Ri(t) triangles
    for i in range(1, 4):
        shifted_Ri = Ri_outlines[i] + displacements[i]
        xr, yr = shifted_Ri[:, 0], shifted_Ri[:, 1]
        frame_data.append(go.Scatter(
            x=np.append(xr, xr[0]), y=np.append(yr, yr[0]),
            mode='lines',
            line=dict(color=colors[i], width=3), # Outline matches the region color
            showlegend=False
        ))

    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

# 3. Build Figure
fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(

    xaxis=dict(range=[-1, 4], scaleanchor="y", scaleratio=1, visible=False),
    yaxis=dict(range=[-1, 4], visible=False),
    paper_bgcolor='white', plot_bgcolor='white',
    updatemenus=[{
        "buttons": [{"args": [None, {"frame": {"duration": 300, "redraw": True}}],
                     "label": "Play", "method": "animate"}],
        "type": "buttons", "x": 0.1, "y": 0
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate"}],
                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],
        "x": 0.1, "len": 0.9
    }],
    showlegend=False, width=800, height=800
)

fig.show()

d=4

In [ ]:
# @title

import plotly.graph_objects as go

import numpy as np

from itertools import permutations



def is_in_R43(x):

    eps = 1e-9

    return (-eps <= x[0] <= x[1] + eps and x[1] <= x[2] + eps and x[2] <= 3 + eps)



def get_minimal_index(vertices):

    vs = {1: [1, 1, 1], 2: [0, 1, 1], 3: [0, 0, 1], 4: [0, 0, 0]}

    for i in range(1, 5):

        v_i = np.array(vs[i])

        if all(is_in_R43(v - v_i) for v in vertices):

            return i

    return 4



# 1. Generate the base cells

cells_data = []

for y1 in range(4):

    for y2 in range(y1, 4):

        for y3 in range(y2, 4):

            base = np.array([y1, y2, y3], dtype=float)

            for p in permutations([0, 1, 2]):

                verts = [base.copy()]

                curr = base.copy()

                for axis in p:

                    curr[axis] += 1

                    verts.append(curr.copy())

                if all(0 <= v[0] <= v[1]+1e-7 and v[1] <= v[2]+1e-7 and v[2] <= 4+1e-7 for v in verts):

                    cells_data.append((np.array(verts), get_minimal_index(verts)))



colors = {1: 'red', 2: 'blue', 3: 'green', 4: 'orange'}



# 2. Define the frames for the animation

t_values = np.linspace(-0.25, 0.25, 23)

frames = []



for t in t_values:

    frame_data = []

    displacements = {

        1: np.array([3*t, 2*t, 1*t]),

        2: np.array([-1*t, 2*t, 1*t]),

        3: np.array([-1*t, -2*t, 1*t]),

        4: np.array([-1*t, -2*t, -3*t])

    }



    for cell, idx in cells_data:

        shifted = cell + displacements[idx]

        x, y, z = shifted[:, 0], shifted[:, 1], shifted[:, 2]



        frame_data.append(go.Mesh3d(

            x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],

            color=colors[idx], opacity=0.4, flatshading=True

        ))



        edge_idx = [0,1,2,0,3,1,3,2]

        frame_data.append(go.Scatter3d(

            x=x[edge_idx], y=y[edge_idx], z=z[edge_idx],

            mode='lines', line=dict(color='black', width=2)

        ))

    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))



# 3. Build the initial figure

fig = go.Figure(data=frames[0].data, frames=frames)



# 4. Add the Slider and Play button with clean layout

fig.update_layout(


    scene=dict(

        xaxis=dict(range=[-1, 5], autorange=False, visible=False, showbackground=False),

        yaxis=dict(range=[-1, 5], autorange=False, visible=False, showbackground=False),

        zaxis=dict(range=[-1, 5], autorange=False, visible=False, showbackground=False),

        aspectmode='cube',

        bgcolor='rgba(0,0,0,0)' # Transparent scene background

    ),

    paper_bgcolor='white',

    updatemenus=[{

        "buttons": [{"args": [None, {"frame": {"duration": 500, "redraw": True}}],

                     "label": "Play", "method": "animate"}],

        "type": "buttons", "showactive": False, "x": 0.1, "y": 0

    }],

    sliders=[{

        "steps": [{"args": [[f.name], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate"}],

                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],

        "transition": {"duration": 300}, "x": 0.1, "len": 0.9

    }],

    showlegend=False

)



fig.show()

d=2

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np

# R_{2,2} is the line [0, 2] subdivided into [0, 1] and [1, 2]
segments = [
    {"coords": np.array([0, 1]), "idx": 2, "color": "blue", "label": "Blue (i=2)"},
    {"coords": np.array([1, 2]), "idx": 1, "color": "red", "label": "Red (i=1)"}
]

# t from -0.5 (overlap) to 0.5 (separated/original R22 shape)
t_values = np.linspace(0.5, -0.5, 21)
frames = []

for t in t_values:
    frame_data = []
    # Displacement difference d1 - d2 = 2t
    d1 = 1 * t
    d2 = -1 * t
    displacements = {1: d1, 2: d2}

    for seg in segments:
        shifted_x = seg["coords"] + displacements[seg["idx"]]
        frame_data.append(go.Scatter(
            x=shifted_x, y=[0, 0],
            mode='lines+markers',
            line=dict(color=seg["color"], width=8),
            marker=dict(size=12, symbol='line-ns-open', line=dict(width=3, color='black')),
            name=seg["label"]
        ))
    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

fig = go.Figure(data=frames[10].data, frames=frames) # Start at t=0

fig.update_layout(

    # Hide axes, grid, and background
    xaxis=dict(
        range=[-1.5, 3.5],
        visible=False
    ),
    yaxis=dict(
        range=[-0.2, 0.2],
        visible=False,
        fixedrange=True
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    updatemenus=[{
        "buttons": [{"args": [None, {"frame": {"duration": 200, "redraw": True}}],
                     "label": "Play", "method": "animate"}],
        "type": "buttons", "x": 0.1, "y": 0
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                   "label": f"{-float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
        "x": 0.1, "len": 0.9
    }],
    showlegend=True,
    width=900,
    height=400
)

fig.show()

d=3

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np

def is_in_R32(x):
    eps = 1e-9
    return (-eps <= x[0] <= x[1] + eps and x[1] <= 2 + eps)

def get_minimal_index(vertices):
    # v1=(1,1), v2=(0,1), v3=(0,0)
    vs = {1: np.array([1, 1]), 2: np.array([0, 1]), 3: np.array([0, 0])}
    for i in range(1, 4):
        if all(is_in_R32(v - vs[i]) for v in vertices):
            return i
    return 3

# 1. Generate the 2D cells (triangles)
cells_data = []
for y1 in range(3):
    for y2 in range(y1, 3):
        # Type 1 triangle: (y1, y2), (y1+1, y2+1), (y1, y2+1)
        t1 = np.array([(y1, y2), (y1+1, y2+1), (y1, y2+1)], dtype=float)
        cells_data.append((t1, get_minimal_index(t1)))
        # Type 2 triangle: (y1, y2), (y1+1, y2), (y1+1, y2+1)
        if y1 + 1 <= y2:
            t2 = np.array([(y1, y2), (y1+1, y2), (y1+1, y2+1)], dtype=float)
            cells_data.append((t2, get_minimal_index(t2)))

colors = {1: 'red', 2: 'blue', 3: 'green'}
t_values = np.linspace(-1/3, 1/3, 21)
frames = []

for t in t_values:
    frame_data = []
    displacements = {
        1: np.array([2*t, 1*t]),
        2: np.array([-1*t, 1*t]),
        3: np.array([-1*t, -2*t])
    }

    for cell, idx in cells_data:
        shifted = cell + displacements[idx]
        x, y = shifted[:, 0], shifted[:, 1]

        frame_data.append(go.Scatter(
            x=np.append(x, x[0]), y=np.append(y, y[0]),
            fill="toself", line=dict(color='black', width=2),
            fillcolor=colors[idx], opacity=0.5, mode='lines'
        ))
    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

# 3. Build Figure
fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(
    # Remove axis lines, labels, and grid
    xaxis=dict(
        range=[-1, 4],
        scaleanchor="y",
        scaleratio=1,
        visible=False
    ),
    yaxis=dict(
        range=[-1, 4],
        visible=False
    ),
    # Set backgrounds to transparent/white
    paper_bgcolor='white',
    plot_bgcolor='white',
    updatemenus=[{
        "buttons": [{"args": [None, {"frame": {"duration": 300, "redraw": True}}],
                     "label": "Play", "method": "animate"}],
        "type": "buttons", "x": 0.1, "y": 0
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate"}],
                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],
        "x": 0.1, "len": 0.9
    }],
    showlegend=False,
    width=800,
    height=800
)

fig.show()

## Reflection of the pyramids across their apex

In [ ]:
# @title

import plotly.graph_objects as go

import numpy as np



def is_in_R32(x):

    eps = 1e-9

    return (-eps <= x[0] <= x[1] + eps and x[1] <= 2 + eps)



def get_minimal_index(vertices_2d):

    vs = {1: np.array([1, 1]), 2: np.array([0, 1]), 3: np.array([0, 0])}

    for i in range(1, 4):

        if all(is_in_R32(v - vs[i]) for v in vertices_2d):

            return i

    return 3



# v_top for R3,3 is (1, 2, 1)

v_top = np.array([1, 2, 1])



# 1. Generate the base 2D cells and their 3D conic extensions

conic_cells = []

for y1 in range(3):

    for y2 in range(y1, 3):

        # Base Triangle Type 1

        t1_base = np.array([(y1, y2), (y1+1, y2+1), (y1, y2+1)], dtype=float)

        idx1 = get_minimal_index(t1_base)

        t1_3d = np.array([[v[0], v[1], 0] for v in t1_base] + [v_top])

        conic_cells.append((t1_3d, idx1))



        # Base Triangle Type 2

        if y1 + 1 <= y2:

            t2_base = np.array([(y1, y2), (y1+1, y2), (y1+1, y2+1)], dtype=float)

            idx2 = get_minimal_index(t2_base)

            t2_3d = np.array([[v[0], v[1], 0] for v in t2_base] + [v_top])

            conic_cells.append((t2_3d, idx2))



colors = {1: 'red', 2: 'blue', 3: 'green'}

t_values = np.linspace(-1/3, 1/3, 15)

frames = []



for t in t_values:

    frame_data = []

    displacements = {

        1: np.array([2*t, 1*t, 0]),

        2: np.array([-1*t, 1*t, 0]),

        3: np.array([-1*t, -2*t, 0])

    }



    for cell, idx in conic_cells:

        # 1. Original shifted piece

        shift_vec = displacements[idx]

        shifted = cell + shift_vec



        # 2. Reflected piece across the apex

        # Current apex position is v_top + shift_vec

        curr_apex = v_top + shift_vec

        reflected = 2 * curr_apex - shifted



        # Plotting logic for both original and reflected

        for pts, opac in [(shifted, 0.5), (reflected, 0.3)]:

            x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]



            # Volume

            frame_data.append(go.Mesh3d(

                x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],

                color=colors[idx], opacity=opac, flatshading=True,

                showlegend=False

            ))

            # Black outlines

            edge_idx = [0,1,2,0,3,1,3,2]

            frame_data.append(go.Scatter3d(

                x=x[edge_idx], y=y[edge_idx], z=z[edge_idx],

                mode='lines', line=dict(color='black', width=1.5),

                showlegend=False

            ))



    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))



fig = go.Figure(data=frames[0].data, frames=frames)



fig.update_layout(

    paper_bgcolor='white',

    scene=dict(

        xaxis=dict(range=[-2, 5], visible=False),

        yaxis=dict(range=[-2, 5], visible=False),

        zaxis=dict(range=[0, 2.5], visible=False),

        aspectmode='cube',

        bgcolor='white'

    ),

    updatemenus=[{

        "buttons": [{"args": [None, {"frame": {"duration": 200, "redraw": True}}],

                     "label": "Play", "method": "animate"}],

        "type": "buttons", "x": 0.1, "y": 0.05

    }],

    sliders=[{

        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],

                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],

        "x": 0.1, "len": 0.9

    }],

    showlegend=False,

    width=900, height=800

)



fig.show()

Reflections of the pyramids across their apex, now including a black wireframe for $C(t)$ and its opposite cone of directions.

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np

def is_in_R32(x):
    eps = 1e-9
    return (-eps <= x[0] <= x[1] + eps and x[1] <= 3 + eps)

def get_minimal_index(vertices_2d):
    vs = {1: np.array([1, 1]), 2: np.array([0, 1]), 3: np.array([0, 0])}
    for i in range(1, 4):
        if all(is_in_R32(v - vs[i]) for v in vertices_2d):
            return i
    return 3

v_top = np.array([1, 2, 1])
v_center = np.array([1, 2, 0])

# The 3x3 base pyramid wireframe points at t=0
base_pyramid_pts = np.array([
    [0, 0, 0],
    [0, 3, 0],
    [3, 3, 0],
    [1, 2, 1]  # Apex is index 3
])

conic_cells = []
for y1 in range(4):
    for y2 in range(y1, 4):
        t1_base = np.array([(y1, y2), (y1+1, y2+1), (y1, y2+1)], dtype=float)
        if all(is_in_R32(v) for v in t1_base):
            idx1 = get_minimal_index(t1_base)
            t1_3d = np.array([[v[0], v[1], 0] for v in t1_base] + [v_top])
            conic_cells.append((t1_3d, idx1))

        if y1 + 1 <= y2:
            t2_base = np.array([(y1, y2), (y1+1, y2), (y1+1, y2+1)], dtype=float)
            if all(is_in_R32(v) for v in t2_base):
                idx2 = get_minimal_index(t2_base)
                t2_3d = np.array([[v[0], v[1], 0] for v in t2_base] + [v_top])
                conic_cells.append((t2_3d, idx2))

colors = {1: 'red', 2: 'blue', 3: 'green'}
t_values = np.linspace(-1/3, 1/3, 15)
frames = []

for t in t_values:
    frame_data = []
    factor = 1 + t

    # 1. Homothety Wireframes
    # Bottom Wireframe (Standard homothety)
    wire_pts = v_center + factor * (base_pyramid_pts - v_center)
    wire_apex = wire_pts[3]

    # Reflected Wireframe (Extension logic)
    # The apex remains the same
    # We calculate the extension for the 3 base points (0, 1, 2)
    s = (2 - wire_apex[2]) / wire_apex[2]

    # Calculate the extended points: P_ext = Apex + s * (Apex - P_base)
    # This keeps the rays perfectly linear without deforming the cone's shape
    refl_base_ext = wire_apex + s * (wire_apex - wire_pts[:3])

    # Combine the extended base with the shared apex
    refl_wire_pts = np.vstack([refl_base_ext, wire_apex])

    for pts in [wire_pts, refl_wire_pts]:
        wx, wy, wz = pts[:, 0], pts[:, 1], pts[:, 2]
        edge_idx = [0, 1, 2, 0, 3, 1, 3, 2]
        frame_data.append(go.Scatter3d(
            x=wx[edge_idx], y=wy[edge_idx], z=wz[edge_idx],
            mode='lines', line=dict(color='black', width=4),
            showlegend=False
        ))

    # 2. Conic Cells (Point-reflected shards)
    displacements = {
        1: np.array([2*t, 1*t, 0]),
        2: np.array([-1*t, 1*t, 0]),
        3: np.array([-1*t, -2*t, 0])
    }

    for cell, idx in conic_cells:
        shift_vec = displacements[idx]
        shifted = cell + shift_vec
        curr_apex = v_top + shift_vec
        reflected = 2 * curr_apex - shifted

        for pts, opac in [(shifted, 0.5), (reflected, 0.3)]:
            x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
            frame_data.append(go.Mesh3d(
                x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
                color=colors[idx], opacity=opac, flatshading=True
            ))
            edge_idx_cell = [0,1,2,0,3,1,3,2]
            frame_data.append(go.Scatter3d(
                x=x[edge_idx_cell], y=y[edge_idx_cell], z=z[edge_idx_cell],
                mode='lines', line=dict(color='black', width=1),
                showlegend=False
            ))

    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-4, 7], visible=False),
        yaxis=dict(range=[-4, 7], visible=False),
        zaxis=dict(range=[-0.1, 2.1], visible=False),
        aspectmode='cube'
    ),
    updatemenus=[{"buttons": [{"args": [None, {"frame": {"duration": 200, "redraw": True}}],
                               "label": "Play", "method": "animate"}],
                  "type": "buttons", "x": 0.1, "y": 0.05}],
    sliders=[{"steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                         "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],
              "x": 0.1, "len": 0.9}],
    width=900, height=800
)

fig.show()

## Regular simplicial subdivision of $R_{3,3}$ with the individual labels of each cell acording to the cited paper

In [ ]:
# @title
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon
import matplotlib.patheffects as path_effects
import itertools

def plot_mirzakhani_subdivision_clean(q=3):
    # Large figsize to provide enough pixel space for massive fonts
    fig, ax = plt.subplots(figsize=(18, 18))
    ax.set_aspect('equal')

    # Remove all axis elements
    ax.set_axis_off()

    cells_data = []
    for w2 in range(q):
        for w1 in range(w2 + 1):
            w = np.array([w1, w2])
            for pi_tuple in [(1, 2), (2, 1)]:
                if w[0] == w[1] and pi_tuple != (1, 2):
                    continue

                pi_inv_1 = pi_tuple.index(1)
                pi_inv_2 = pi_tuple.index(2)

                v0 = w.astype(float)
                v1 = v0.copy()
                v1[pi_inv_2] += 1
                v2 = v1.copy()
                v2[pi_inv_1] += 1

                if all(p[0] <= p[1] for p in [v0, v1, v2]):
                    cells_data.append((np.array([v0, v1, v2]), w, pi_tuple))

    # Plotting loop
    num_cells = len(cells_data)
    for i, (verts, w, pi) in enumerate(cells_data):
        poly = Polygon(verts, facecolor=plt.cm.Pastel1(i/num_cells), edgecolor='black', alpha=0.4)
        ax.add_patch(poly)

        centroid = np.mean(verts, axis=0)

        pi_label = r"\text{id}" if pi == (1, 2) else r"\tau"
        # Using a more compact representation to allow larger font scaling
        label = r"$\sigma(({0},{1}), {2})$".format(int(w[0]), int(w[1]), pi_label)

        # Fontsize increased to 32 to fill the cell area
        txt = ax.text(centroid[0], centroid[1], label, ha='center', va='center',
                      fontsize=32, fontweight='bold')

        # Heavy white outline to handle the large font overlapping cell edges
        txt.set_path_effects([path_effects.withStroke(linewidth=7, foreground='white')])

    # Wireframes for the 3 v_i + R_{3,2} pieces
    v_vectors = [np.array([1, 1]), np.array([0, 1]), np.array([0, 0])]
    colors = ['#e41a1c', '#377eb8', '#4daf4a']
    r32_base = np.array([[0, 0], [0, 2], [2, 2], [0, 0]])

    for i, (v, color) in enumerate(zip(v_vectors, colors), 1):
        wireframe = r32_base + v
        ax.plot(wireframe[:, 0], wireframe[:, 1], color=color, lw=8,
                label=rf"$\mathbf{{v}}_{i} + R_{{3,2}}$", zorder=10)

    # Outer boundary
    ax.plot([0, 0, q, 0], [0, q, q, 0], color='black', lw=4, zorder=5)


    plt.tight_layout()
    plt.show()

plot_mirzakhani_subdivision_clean(3)

In [ ]:
# @title

import plotly.graph_objects as go
import numpy as np
from itertools import permutations

def plot_simplicial_3d(q=3):
    # v vectors for d=4
    vs = {
        1: np.array([1, 1, 1]),
        2: np.array([0, 1, 1]),
        3: np.array([0, 0, 1]),
        4: np.array([0, 0, 0])
    }

    cells_data = []
    # Generate all cells sigma(w, pi)
    # w in W_{4, q-1}
    for w3 in range(q):
        for w2 in range(w3 + 1):
            for w1 in range(w2 + 1):
                base = np.array([w1, w2, w3], dtype=float)
                # Check all 3! permutations
                for p in permutations([0, 1, 2]):
                    # Consistency check (Simplified for visualization)
                    verts = [base.copy()]
                    curr = base.copy()
                    # The formula creates a sequence of vertices
                    for axis in p:
                        curr[axis] += 1
                        verts.append(curr.copy())

                    # Ensure the cell is inside R_{4,q} (y1 <= y2 <= y3 <= q)
                    if all(0 <= v[0] <= v[1]+1e-7 and v[1] <= v[2]+1e-7 and v[2] <= q+1e-7 for v in verts):
                        cells_data.append(np.array(verts))

    fig = go.Figure()

    # 1. Plot each cell with a unique color
    # Generate a spectrum of colors
    num_cells = len(cells_data)
    colors = [f'hsl({int(360*i/num_cells)}, 70%, 50%)' for i in range(num_cells)]

    for i, cell in enumerate(cells_data):
        x, y, z = cell[:, 0], cell[:, 1], cell[:, 2]

        # Mesh for the solid tetrahedron
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            # Tetrahedral triangulation indices
            i=[0, 0, 0, 1], j=[1, 1, 2, 2], k=[2, 3, 3, 3],
            color=colors[i],
            opacity=0.3,
            flatshading=True,
            hoverinfo='none'
        ))

        # Black wireframe for the individual cell edges
        edge_idx = [0, 1, 2, 0, 3, 1, 3, 2]
        fig.add_trace(go.Scatter3d(
            x=x[edge_idx], y=y[edge_idx], z=z[edge_idx],
            mode='lines',
            line=dict(color='black', width=1),
            hoverinfo='none'
        ))

    # 2. Plot the 4 large Corner Wireframes (v_i + R_{4,2})
    # R_{4,2} vertices: (0,0,0), (0,0,2), (0,2,2), (2,2,2)
    r42_base_verts = np.array([
        [0, 0, 0],
        [0, 0, 2],
        [0, 2, 2],
        [2, 2, 2]
    ])
    # Edges for a tetrahedron: (0,1), (1,2), (2,0), (0,3), (1,3), (2,3)
    wire_edges = [0, 1, 2, 0, 3, 1, 3, 2]
    corner_colors = ['red', 'blue', 'green', 'orange']

    for i, (key, v_vec) in enumerate(vs.items()):
        shifted_r42 = r42_base_verts + v_vec
        fig.add_trace(go.Scatter3d(
            x=shifted_r42[wire_edges, 0],
            y=shifted_r42[wire_edges, 1],
            z=shifted_r42[wire_edges, 2],
            mode='lines',
            line=dict(color=corner_colors[i], width=6),
            name=f'v_{key} + R_{{4,2}}'
        ))

    # 3. Layout adjustments
    fig.update_layout(
        showlegend=False,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        paper_bgcolor='white'
    )

    fig.show()

plot_simplicial_3d(q=3)

## Base case for the Perron tree construction

d=3

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np
from itertools import permutations

def is_in_R32(x):
    eps = 1e-9
    return (-eps <= x[0] <= x[1] + eps and x[1] <= 2 + eps)

def get_minimal_index(vertices_2d):
    vs = {1: np.array([1, 1]), 2: np.array([0, 1]), 3: np.array([0, 0])}
    for i in range(1, 4):
        if all(is_in_R32(v - vs[i]) for v in vertices_2d):
            return i
    return 3

# v_top for R3,3 is (1, 2, 1)
v_top = np.array([1, 2, 1])

# 1. Generate the base 2D cells and their 3D conic extensions
conic_cells = []
for y1 in range(3):
    for y2 in range(y1, 3):
        # Base Triangle Type 1
        t1_base = np.array([(y1, y2), (y1+1, y2+1), (y1, y2+1)], dtype=float)
        idx1 = get_minimal_index(t1_base)
        # Form tetrahedron by adding v_top (appending z=0 to base first)
        t1_3d = np.array([[v[0], v[1], 0] for v in t1_base] + [v_top])
        conic_cells.append((t1_3d, idx1))

        # Base Triangle Type 2
        if y1 + 1 <= y2:
            t2_base = np.array([(y1, y2), (y1+1, y2), (y1+1, y2+1)], dtype=float)
            idx2 = get_minimal_index(t2_base)
            t2_3d = np.array([[v[0], v[1], 0] for v in t2_base] + [v_top])
            conic_cells.append((t2_3d, idx2))

colors = {1: 'red', 2: 'blue', 3: 'green'}
t_values = np.linspace(-1/3, 1/3, 15)
frames = []

for t in t_values:
    frame_data = []
    # Displacements
    displacements = {
        1: np.array([2*t, 1*t, 0]),
        2: np.array([-1*t, 1*t, 0]),
        3: np.array([-1*t, -2*t, 0])
    }

    for cell, idx in conic_cells:
        shifted = cell + displacements[idx]
        x, y, z = shifted[:, 0], shifted[:, 1], shifted[:, 2]

        # Add volume
        frame_data.append(go.Mesh3d(
            x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
            color=colors[idx], opacity=0.5, flatshading=True,
            showlegend=False
        ))
        # Add black outlines
        edge_idx = [0,1,2,0,3,1,3,2]
        frame_data.append(go.Scatter3d(
            x=x[edge_idx], y=y[edge_idx], z=z[edge_idx],
            mode='lines', line=dict(color='black', width=2),
            showlegend=False
        ))
    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(
    paper_bgcolor='white',
    scene=dict(
        xaxis=dict(range=[-1, 4], visible=False, showbackground=False),
        yaxis=dict(range=[-1, 4], visible=False, showbackground=False),
        zaxis=dict(range=[0, 2], visible=False, showbackground=False),
        aspectmode='cube',
        bgcolor='white'
    ),
    updatemenus=[{
        "buttons": [{"args": [None, {"frame": {"duration": 200, "redraw": True}}],
                     "label": "Play", "method": "animate"}],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                   "label": f"{-t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)],
        "x": 0.1, "len": 0.9
    }],
    showlegend=False,
    width=900, height=800
)

fig.show()

d=2

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np

# Base segments for R2,2
# Red: [1, 2], Blue: [0, 1]
# v_top = (1, 1)
v_top = np.array([1, 1])

t_values = np.linspace(-1/2, 0.5, 11)
frames = []

for t in t_values:
    frame_data = []
    # k=2 displacements: d1 = 2t, d2 = -2t (applied only to x)
    d1 = np.array([2*t, 0])
    d2 = np.array([-2*t, 0])

    # Define the two triangles (base segment + v_top)
    # Blue Triangle: (0,0), (1,0), (1,1)
    blue_tri = np.array([[0, 0], [1, 0], [1, 1]]) + d2
    # Red Triangle: (1,0), (2,0), (1,1) + d1
    red_tri = np.array([[1, 0], [2, 0], [1, 1]]) + d1

    for tri, color, label in [(blue_tri, 'blue', 'Index 2'), (red_tri, 'red', 'Index 1')]:
        x, y = tri[:, 0], tri[:, 1]
        frame_data.append(go.Scatter(
            x=np.append(x, x[0]), y=np.append(y, y[0]),
            fill="toself", line=dict(color='black', width=2),
            fillcolor=color, opacity=0.5, name=label
        ))

    frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(

    # Hide axes and background
    xaxis=dict(
        range=[-2, 4],
        visible=False
    ),
    yaxis=dict(
        range=[-0.5, 2],
        scaleanchor="x",
        scaleratio=1,
        visible=False
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 300, "redraw": True}}],
             "label": "Play", "method": "animate"}
        ],
        "type": "buttons",
        "x": 0.1,
        "y": 0
    }],
    sliders=[{
        "steps": [
            {"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
             "label": f"{-float(f.name.split('=')[1]):.2f}", "method": "animate"}
            for f in frames
        ],
        "x": 0.1,
        "len": 0.9
    }],
    showlegend=True,
    width=800,
    height=500
)

fig.show()

## First iteration of the Perron tree construction for $d=3$, $n=2$ and $t_1=1/4$

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# PARAMETER
# ============================================================
t = 1/3

# ============================================================
# BASE SIMPLEX
# ============================================================
P1 = np.array([
    [0,0,0],
    [0,3,0],
    [3,3,0],
    [1,2,1]
], dtype=float)

b = np.array([1.0,2.0])
v = {
    1: np.array([1.0,1.0]),
    2: np.array([0.0,1.0]),
    3: np.array([0.0,0.0])
}

# ============================================================
# SUBDIVISION
# ============================================================
def base_triangles():
    tris=[]
    for a in range(3):
        for b_ in range(a,3):
            if a==b_:
                tris.append([(a,b_),(a,b_+1),(a+1,b_+1)])
            else:
                tris.append([(a,b_),(a,b_+1),(a+1,b_+1)])
                tris.append([(a,b_),(a+1,b_),(a+1,b_+1)])
    return tris

def classify(tri):
    for j in [1,2,3]:
        ok=True
        for x,y in tri:
            if j==1 and not (1<=x<=y<=3): ok=False
            if j==2 and not (0<=x<=y-1 and 1<=y<=3): ok=False
            if j==3 and not (0<=x<=y<=2): ok=False
        if ok: return j
    return 3

tris = base_triangles()
classes = [classify(tri) for tri in tris]

# ============================================================
# BASE CONSTRUCTION
# ============================================================
canonical_cells = []

for tri, j in zip(tris, classes):
    tri3 = np.array([[x,y,0] for x,y in tri])
    apex = np.array([1,2,1])

    shift = np.array([*(t*(b-3*v[j])),0])
    tet = np.vstack([tri3,apex]) + shift
    canonical_cells.append(tet)

# ============================================================
# FIRST LEVEL CELLS
# ============================================================
T_cells = []
for tri in tris:
    tri3 = np.array([[x,y,0] for x,y in tri])
    T_cells.append(np.vstack([tri3,[1,2,1]]))

# ============================================================
# AFFINE MAP
# ============================================================
def affine_map(P_from, P_to):
    A = np.zeros((3,3))
    for k in range(3):
        A[:,k] = P_to[k+1] - P_to[0]

    A0 = np.zeros((3,3))
    for k in range(3):
        A0[:,k] = P_from[k+1] - P_from[0]

    M = A @ np.linalg.inv(A0)
    b_vec = P_to[0] - M @ P_from[0]

    return lambda x: (M @ x.T).T + b_vec

# ============================================================
# PLOTTING HELPERS
# ============================================================
def tetra_faces(vertices, color, opacity):
    vtx=np.array(vertices)
    i,j,k=zip(*[(0,1,2),(0,1,3),(0,2,3),(1,2,3)])
    return go.Mesh3d(
        x=vtx[:,0], y=vtx[:,1], z=vtx[:,2],
        i=i,j=j,k=k,
        color=color,
        opacity=opacity,
        flatshading=True,
        hoverinfo='skip'
    )

def tetra_edges(vertices):
    vtx=np.array(vertices)
    edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
    xs,ys,zs=[],[],[]
    for a,b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]

    return go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(color="black", width=3),
        showlegend=False,
        hoverinfo='skip'
    )

# ============================================================
# BUILD FIGURE
# ============================================================
fig = go.Figure()

# Outer simplex
fig.add_trace(tetra_faces(P1,"lightgray",0.05))
fig.add_trace(tetra_edges(P1))

colors = ["orange","purple","green"]

# ============================================================
# SECOND LEVEL (81 CELLS)
# ============================================================
for T in T_cells:
    A = affine_map(P1, T)

    for k, cell in enumerate(canonical_cells):
        mapped = A(cell)
        fig.add_trace(tetra_faces(mapped, colors[k%3], 0.35))
        fig.add_trace(tetra_edges(mapped))

# ============================================================
# LAYOUT (CLEAN)
# ============================================================
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False, showbackground=False),
        yaxis=dict(visible=False, showbackground=False),
        zaxis=dict(visible=False, showbackground=False),
        aspectmode="data",
        camera=dict(eye=dict(x=1.7,y=1.7,z=1.2)),
        bgcolor='white'
    ),
    paper_bgcolor='white',
    width=1000,
    height=850,
    margin=dict(l=0, r=0, b=0, t=40),

)

fig.show()

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
v_map = {
    1: np.array([1.0, 1.0]),
    2: np.array([0.0, 1.0]),
    3: np.array([0.0, 0.0])
}

# ============================================================
# SUBDIVISION LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            # Triangle Type 1
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            # Triangle Type 2
            if a != b_:
                tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x, y in tri:
            if j == 1 and not (1 <= x <= y <= 3): ok = False
            if j == 2 and not (0 <= x <= y - 1 and 1 <= y <= 3): ok = False
            if j == 3 and not (0 <= x <= y <= 2): ok = False
        if ok: return j
    return 3

tris = get_base_triangles()
classes = [classify(tri) for tri in tris]
parent_simplices = [np.vstack([np.array([[x, y, 0] for x, y in tri]), apex_main]) for tri in tris]

# ============================================================
# AFFINE MAP CALCULATION
# ============================================================
def get_affine_params(P_from, P_to):
    A0 = np.zeros((3, 3))
    A_target = np.zeros((3, 3))
    for k in range(3):
        A0[:, k] = P_from[k+1] - P_from[0]
        A_target[:, k] = P_to[k+1] - P_to[0]
    M = A_target @ np.linalg.inv(A0)
    translation = P_to[0] - M @ P_from[0]
    return M, translation

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

# ============================================================
# WIREFRAME HELPERS
# ============================================================
def get_tet_wireframe(pts):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    wx, wy, wz = [], [], []
    for i, j in edges:
        wx.extend([pts[i,0], pts[j,0], None])
        wy.extend([pts[i,1], pts[j,1], None])
        wz.extend([pts[i,2], pts[j,2], None])
    return wx, wy, wz

# ============================================================
# STATIC BASE FRAME (WIRE)
# ============================================================
base_lines_x, base_lines_y, base_lines_z = [], [], []
for tri in tris:
    pts = tri + [tri[0]]
    for p in pts:
        base_lines_x.append(p[0])
        base_lines_y.append(p[1])
        base_lines_z.append(0)
    base_lines_x.append(None)
    base_lines_y.append(None)
    base_lines_z.append(None)

base_wireframe = go.Scatter3d(
    x=base_lines_x, y=base_lines_y, z=base_lines_z,
    mode='lines',
    line=dict(color='black', width=4),
    name='Parent Grid',
    showlegend=True
)

# ============================================================
# FRAME GENERATION
# ============================================================
t_steps = np.linspace(0, 1/3, 20)
colors_list = ["orange", "purple", "green"]
dark_colors = ["darkorange", "indigo", "darkgreen"]

def create_frame(t):
    frame_data = []
    canonical_cells = []
    for tri, j in zip(tris, classes):
        tri3 = np.array([[x, y, 0] for x, y in tri])
        shift = np.array([*(t * (b_vec_2d - 3 * v_map[j])), 0])
        tet = np.vstack([tri3, apex_main]) + shift
        canonical_cells.append((tet, j))

    for M, trans in parent_maps:
        for cell_pts, color_idx in canonical_cells:
            mapped = (M @ cell_pts.T).T + trans
            frame_data.append(go.Mesh3d(
                x=mapped[:,0], y=mapped[:,1], z=mapped[:,2],
                i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
                color=colors_list[color_idx-1],
                opacity=0.5,
                flatshading=True,
                showlegend=False
            ))
            wx, wy, wz = get_tet_wireframe(mapped)
            frame_data.append(go.Scatter3d(
                x=wx, y=wy, z=wz,
                mode='lines',
                line=dict(color=dark_colors[color_idx-1], width=2),
                showlegend=False
            ))
    return frame_data

# ============================================================
# BUILD FIGURE
# ============================================================
frames = [go.Frame(data=create_frame(t), name=f"t={t:.3f}") for t in t_steps]

fig = go.Figure(
    data=frames[0].data,
    layout=go.Layout(

        paper_bgcolor='white',
        plot_bgcolor='white',
        scene=dict(
            xaxis=dict(visible=False, showbackground=False),
            yaxis=dict(visible=False, showbackground=False),
            zaxis=dict(visible=False, showbackground=False),
            aspectmode="data"
        ),
        updatemenus=[{
            "buttons": [
                {"args": [None, {"frame": {"duration": 100, "redraw": True}}], "label": "Play", "method": "animate"},
                {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": "Pause", "method": "animate"}
            ],
            "type": "buttons", "x": 0.1, "y": 0
        }],
        sliders=[{
            "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                       "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
            "x": 0.1, "len": 0.9
        }]
    ),
    frames=frames
)

fig.add_trace(go.Mesh3d(
    x=P1[:,0], y=P1[:,1], z=P1[:,2],
    i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
    color="lightgray", opacity=0.05, name="Main Simplex"
))
fig.add_trace(base_wireframe)

fig.show()

## Core aligment before beginning the second iteration of the construction

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
global_barycenter = np.array([1.0, 2.0, 0.0])
homothetic_factor = 0.5

v_map = {1: np.array([1.0, 1.0]), 2: np.array([0.0, 1.0]), 3: np.array([0.0, 0.0])}
colors_list = ["orange", "purple", "green"]
wire_colors = ["darkorange", "indigo", "darkgreen"]

# ============================================================
# WIREFRAME HELPER
# ============================================================
def get_skeleton(vtx):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]
    return xs, ys, zs

orig_x, orig_y, orig_z = get_skeleton(P1)
H_simplex = global_barycenter + homothetic_factor * (P1 - global_barycenter)
homo_x, homo_y, homo_z = get_skeleton(H_simplex)

# ============================================================
# SUBDIVISION LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            if a != b_: tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x, y in tri:
            if j == 1 and not (1 <= x <= y <= 3): ok = False
            if j == 2 and not (0 <= x <= y - 1 and 1 <= y <= 3): ok = False
            if j == 3 and not (0 <= x <= y <= 2): ok = False
        if ok: return j
    return 3

tris = get_base_triangles()
classes = [classify(tri) for tri in tris]
parent_simplices = [np.vstack([np.array([[x, y, 0] for x, y in tri]), apex_main]) for tri in tris]

def get_affine_params(P_from, P_to):
    A0 = np.zeros((3, 3))
    A_target = np.zeros((3, 3))
    for k in range(3):
        A0[:, k] = P_from[k+1] - P_from[0]
        A_target[:, k] = P_to[k+1] - P_to[0]
    M = A_target @ np.linalg.inv(A0)
    return M, P_to[0] - M @ P_from[0]

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

t_fixed = 1/4
frozen_children = []
for tri, j in zip(tris, classes):
    tri3 = np.array([[x, y, 0] for x, y in tri])
    shift = np.array([*(t_fixed * (b_vec_2d - 3 * v_map[j])), 0])
    frozen_children.append(np.vstack([tri3, apex_main]) + shift)

# ============================================================
# GLOBAL SHIFT LOGIC (p)
# ============================================================
p_steps = np.linspace(0, 0.25, 25)

def create_p_frame(p):
    meshes = []
    wires_coords = {0: [[], [], []], 1: [[], [], []], 2: [[], [], []]}

    for i, (M, trans) in enumerate(parent_maps):
        parent_base_center = np.mean(parent_simplices[i][:3], axis=0)
        global_shift_vec = p * (global_barycenter - parent_base_center)

        for k, child_cell in enumerate(frozen_children):
            color_idx = k % 3
            mapped = (M @ child_cell.T).T + trans
            final_pos = mapped + global_shift_vec

            meshes.append(go.Mesh3d(
                x=final_pos[:,0], y=final_pos[:,1], z=final_pos[:,2],
                i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
                color=colors_list[color_idx], opacity=0.5, flatshading=True,
                showlegend=False
            ))

            wx, wy, wz = get_skeleton(final_pos)
            wires_coords[color_idx][0].extend(wx)
            wires_coords[color_idx][1].extend(wy)
            wires_coords[color_idx][2].extend(wz)

    wire_traces = []
    for idx in range(3):
        wire_traces.append(go.Scatter3d(
            x=wires_coords[idx][0], y=wires_coords[idx][1], z=wires_coords[idx][2],
            mode='lines', line=dict(color=wire_colors[idx], width=2), showlegend=False
        ))

    return meshes + wire_traces

# ============================================================
# BUILD FIGURE
# ============================================================
frames = [go.Frame(data=create_p_frame(p), name=f"p={p:.2f}") for p in p_steps]
fig = go.Figure(data=frames[0].data, frames=frames)

fig.add_trace(go.Scatter3d(
    x=orig_x, y=orig_y, z=orig_z, mode='lines',
    line=dict(color='blue', width=5), name="Original Skeleton"
))

fig.add_trace(go.Scatter3d(
    x=homo_x, y=homo_y, z=homo_z, mode='lines',
    line=dict(color='black', width=5), name="Target Skeleton"
))

fig.update_layout(

    paper_bgcolor='white',
    scene=dict(
        xaxis=dict(range=[-0.5, 3.5], visible=False, showbackground=False),
        yaxis=dict(range=[-0.5, 3.5], visible=False, showbackground=False),
        zaxis=dict(range=[0, 1.2], visible=False, showbackground=False),
        aspectmode="data",
        bgcolor='rgba(0,0,0,0)'
    ),
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 100, "redraw": True}}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": "Pause", "method": "animate"}
        ], "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                   "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
        "x": 0.1, "len": 0.9, "currentvalue": {"prefix": "Global Shift p: "}
    }],
    width=1000, height=800
)

fig.show()

## Second iteration with cores of previous iteration as our starting point

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
global_barycenter = np.array([1.0, 2.0, 0.0])
homothetic_factor = 0.5

v_map = {1: [1,1], 2: [0,1], 3: [0,0]}
wire_colors = {1: "darkorange", 2: "indigo", 3: "darkgreen"}

# ============================================================
# WIREFRAME HELPER
# ============================================================
def get_simplex_skeleton(vtx):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]
    return xs, ys, zs

orig_x, orig_y, orig_z = get_simplex_skeleton(P1)
H_simplex = global_barycenter + homothetic_factor * (P1 - global_barycenter)
homo_x, homo_y, homo_z = get_simplex_skeleton(H_simplex)

# ============================================================
# SUBDIVISION & MAPPING LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            if a != b_: tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x,y in tri:
            if j==1 and not (1<=x<=y<=3): ok=False
            if j==2 and not (0<=x<=y-1 and 1<=y<=3): ok=False
            if j==3 and not (0<=x<=y<=2): ok=False
        if ok: return j
    return 3

tris = get_base_triangles()
parent_classes = [classify(t) for t in tris]
parent_simplices = [np.vstack([np.array([[x,y,0] for x,y in t]), apex_main]) for t in tris]

def get_affine_params(P_from, P_to):
    A_target, A_orig = np.zeros((3,3)), np.zeros((3,3))
    for k in range(3):
        A_target[:,k] = P_to[k+1] - P_to[0]
        A_orig[:,k] = P_from[k+1] - P_from[0]
    M = A_target @ np.linalg.inv(A_orig)
    return M, P_to[0] - M @ P_from[0]

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

t_fixed = 1/4
frozen_children = []
for tri in tris:
    j = classify(tri)
    c = np.vstack([np.array([[x,y,0] for x,y in tri]), apex_main])
    shift = np.array([*(t_fixed*(b_vec_2d - 3*np.array(v_map[j]))), 0])
    frozen_children.append(c + shift)

# ============================================================
# ANIMATION & FIGURE ASSEMBLY
# ============================================================
p_fixed = 0.25
s_steps = np.linspace(0, 0.25, 20)
colors_list = {1: "orange", 2: "purple", 3: "green"}

def create_frame_data(s):
    meshes = []
    wires_coords = {1: [[], [], []], 2: [[], [], []], 3: [[], [], []]}

    for i, (M, trans) in enumerate(parent_maps):
        j_p = parent_classes[i]
        bary_shift = p_fixed * (global_barycenter - np.mean(parent_simplices[i][:3], 0))
        block_shift = np.array([*(s * (b_vec_2d - 3 * np.array(v_map[j_p]))), 0])
        total_shift = bary_shift + block_shift

        for child in frozen_children:
            pos = (M @ child.T).T + trans + total_shift

            meshes.append(go.Mesh3d(
                x=pos[:,0], y=pos[:,1], z=pos[:,2],
                i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
                color=colors_list[j_p], opacity=0.6, flatshading=True,
                showlegend=False
            ))

            wx, wy, wz = get_simplex_skeleton(pos)
            wires_coords[j_p][0].extend(wx)
            wires_coords[j_p][1].extend(wy)
            wires_coords[j_p][2].extend(wz)

    wire_traces = []
    for c_idx, coords in wires_coords.items():
        wire_traces.append(go.Scatter3d(
            x=coords[0], y=coords[1], z=coords[2],
            mode='lines',
            line=dict(color=wire_colors[c_idx], width=2),
            showlegend=False
        ))

    return meshes + wire_traces

frames = [go.Frame(data=create_frame_data(s), name=f"s={s:.2f}") for s in s_steps]
fig = go.Figure(data=frames[0].data, frames=frames)

fig.add_trace(go.Scatter3d(
    x=orig_x, y=orig_y, z=orig_z, mode='lines',
    line=dict(color='blue', width=5), name="Original Skeleton"
))

fig.add_trace(go.Scatter3d(
    x=homo_x, y=homo_y, z=homo_z, mode='lines',
    line=dict(color='black', width=5), name="Target Skeleton"
))

# --- Layout ---
fig.update_layout(

    paper_bgcolor='white',
    scene=dict(
        xaxis=dict(range=[-0.5, 3.5], visible=False, showbackground=False),
        yaxis=dict(range=[-0.5, 3.5], visible=False, showbackground=False),
        zaxis=dict(range=[0, 1.1], visible=False, showbackground=False),
        aspectmode="data"
    ),
    width=1000, height=800,
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 50, "redraw": True}}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}}], "label": "Pause", "method": "animate"}
        ],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}}],
                   "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
        "x": 0.1, "len": 0.9
    }]
)

fig.show()

Added reflected pyramids and the opposite cone of the resulting core

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
global_barycenter = np.array([1.0, 2.0, 0.0])

v_map = {1: [1,1], 2: [0,1], 3: [0,0]}
wire_colors = {1: "darkorange", 2: "indigo", 3: "darkgreen"}
colors_list = {1: "orange", 2: "purple", 3: "green"}

# ============================================================
# WIREFRAME HELPERS
# ============================================================
def get_simplex_skeleton(vtx):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]
    return xs, ys, zs

def get_extended_cone(target_vtx, target_apex, z_target=2.0):
    xs, ys, zs = [], [], []
    for i in range(3):
        n = (i + 1) % 3
        xs += [target_vtx[i,0], target_vtx[n,0], None]
        ys += [target_vtx[i,1], target_vtx[n,1], None]
        zs += [target_vtx[i,2], target_vtx[n,2], None]
    v_base = target_vtx[:3] - target_apex
    scale_ratio = (z_target - target_apex[2]) / (target_vtx[0,2] - target_apex[2])
    extended_base = target_apex + v_base * scale_ratio
    for i in range(3):
        n = (i + 1) % 3
        xs += [extended_base[i,0], extended_base[n,0], None]
        ys += [extended_base[i,1], extended_base[n,1], None]
        zs += [extended_base[i,2], extended_base[n,2], None]
    for i in range(3):
        xs += [target_vtx[i,0], extended_base[i,0], None]
        ys += [target_vtx[i,1], extended_base[i,1], None]
        zs += [target_vtx[i,2], extended_base[i,2], None]
    return xs, ys, zs

orig_x, orig_y, orig_z = get_simplex_skeleton(P1)

# ============================================================
# SUBDIVISION & MAPPING LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            if a != b_: tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x,y in tri:
            if j==1 and not (1<=x<=y<=3): ok=False
            if j==2 and not (0<=x<=y-1 and 1<=y<=3): ok=False
            if j==3 and not (0<=x<=y<=2): ok=False
        if ok: return j
    return 3

tris = get_base_triangles()
parent_classes = [classify(t) for t in tris]
parent_simplices = [np.vstack([np.array([[x,y,0] for x,y in t]), apex_main]) for t in tris]

def get_affine_params(P_from, P_to):
    A_target, A_orig = np.zeros((3,3)), np.zeros((3,3))
    for k in range(3):
        A_target[:,k] = P_to[k+1] - P_to[0]
        A_orig[:,k] = P_from[k+1] - P_from[0]
    M = A_target @ np.linalg.inv(A_orig)
    return M, P_to[0] - M @ P_from[0]

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

t_fixed = 1/4
frozen_children = []
for tri in tris:
    j = classify(tri)
    c = np.vstack([np.array([[x,y,0] for x,y in tri]), apex_main])
    shift = np.array([*(t_fixed*(b_vec_2d - 3*np.array(v_map[j]))), 0])
    frozen_children.append(c + shift)

# ============================================================
# ANIMATION & FIGURE ASSEMBLY
# ============================================================
p_fixed = 0.25
s_steps = np.linspace(0, 0.25, 25)

def create_frame_data(s):
    meshes = []
    wires_coords = {1: [[], [], []], 2: [[], [], []], 3: [[], [], []]}

    # --- DYNAMIC CORE CALCULATION ---
    # Resulting core homothety: (1 - 1/4)(1 - s)
    q = s
    dynamic_homo_factor = (1 - 0.25) * (1 - q*4/3)
    H_dynamic = global_barycenter + dynamic_homo_factor * (P1 - global_barycenter)
    h_x, h_y, h_z = get_simplex_skeleton(H_dynamic)

    # --- DYNAMIC CONE CALCULATION ---
    c_x, c_y, c_z = get_extended_cone(H_dynamic, H_dynamic[3], z_target=2.0)

    # Calculate Migrating Simplices
    for i, (M, trans) in enumerate(parent_maps):
        j_p = parent_classes[i]
        bary_shift = p_fixed * (global_barycenter - np.mean(parent_simplices[i][:3], 0))
        block_shift = np.array([*(s * (b_vec_2d - 3 * np.array(v_map[j_p]))), 0])
        total_shift = bary_shift + block_shift

        for child in frozen_children:
            pos = (M @ child.T).T + trans + total_shift
            meshes.append(go.Mesh3d(x=pos[:,0], y=pos[:,1], z=pos[:,2], i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3], color=colors_list[j_p], opacity=0.6, flatshading=True))
            wx, wy, wz = get_simplex_skeleton(pos)
            wires_coords[j_p][0].extend(wx); wires_coords[j_p][1].extend(wy); wires_coords[j_p][2].extend(wz)

            # Reflection logic
            local_apex = pos[3]
            reflected_pos = 2 * local_apex - pos
            meshes.append(go.Mesh3d(x=reflected_pos[:,0], y=reflected_pos[:,1], z=reflected_pos[:,2], i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3], color=colors_list[j_p], opacity=0.3, flatshading=True))
            rwx, rwy, rwz = get_simplex_skeleton(reflected_pos)
            wires_coords[j_p][0].extend(rwx); wires_coords[j_p][1].extend(rwy); wires_coords[j_p][2].extend(rwz)

    # Build the traces for this specific frame
    frame_traces = []
    for c_idx, coords in wires_coords.items():
        frame_traces.append(go.Scatter3d(x=coords[0], y=coords[1], z=coords[2], mode='lines', line=dict(color=wire_colors[c_idx], width=1.5), showlegend=False))

    # ADD CORE AND CONE TO EVERY FRAME
    core_trace = go.Scatter3d(x=h_x, y=h_y, z=h_z, mode='lines', line=dict(color='black', width=4), name="Resulting core")
    cone_trace = go.Scatter3d(x=c_x, y=c_y, z=c_z, mode='lines', line=dict(color='blue', width=2), name="Reflected cone")

    return meshes + frame_traces + [core_trace, cone_trace]

frames = [go.Frame(data=create_frame_data(s), name=f"s={s:.2f}") for s in s_steps]

# Initialize figure with the first frame's data
fig = go.Figure(data=frames[0].data, frames=frames)

# Only add the "Original Simplex" as a static trace
fig.add_trace(go.Scatter3d(x=orig_x, y=orig_y, z=orig_z, mode='lines', line=dict(color='lightgrey', width=2), name="Original simplex"))

fig.update_layout(

    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False, range=[0, 2.2]), aspectmode="data", bgcolor="white"),
    width=1000, height=800,
    updatemenus=[{"buttons": [{"args": [None, {"frame": {"duration": 50, "redraw": True}}], "label": "Play", "method": "animate"}], "type": "buttons", "x": 0.1, "y": 0.05}],
    sliders=[{"steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}}], "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames], "x": 0.1, "len": 0.9}]
)

fig.show()

## $\delta$-tubes inscribed inside of each pyramid in the Perron tree construction

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
global_barycenter = np.array([1.0, 2.0, 0.0])
homothetic_factor = 0.5

v_map = {1: [1,1], 2: [0,1], 3: [0,0]}
wire_colors = {1: "darkorange", 2: "indigo", 3: "darkgreen"}

# ============================================================
# HELPERS: WIREFRAME & ROTATED TUBE
# ============================================================
def get_simplex_skeleton(vtx):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]
    return xs, ys, zs

def get_rotated_tube(pts, radius=1/36, height_factor=0.75, n_sides=12):
    base_center = np.mean(pts[:3], axis=0)
    apex = pts[3]
    vec = apex - base_center
    length = np.linalg.norm(vec)
    if length < 1e-6: return None
    direction = vec / length

    h = length * height_factor
    theta = np.linspace(0, 2*np.pi, n_sides, endpoint=False)
    cx, cy = radius * np.cos(theta), radius * np.sin(theta)

    v_bot = np.column_stack([cx, cy, np.zeros(n_sides)])
    v_top = np.column_stack([cx, cy, np.full(n_sides, h)])
    verts = np.vstack([v_bot, v_top])

    z_axis = np.array([0, 0, 1])
    if np.allclose(direction, z_axis): R = np.eye(3)
    elif np.allclose(direction, -z_axis): R = np.diag([1, -1, -1])
    else:
        v = np.cross(z_axis, direction)
        s, c = np.linalg.norm(v), np.dot(z_axis, direction)
        v_skew = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + v_skew + (v_skew @ v_skew) * ((1 - c) / (s**2))

    verts = (R @ verts.T).T + base_center
    idx_i, idx_j, idx_k = [], [], []
    for s in range(n_sides):
        ns = (s + 1) % n_sides
        idx_i.extend([s, s, ns + n_sides])
        idx_j.extend([ns, ns + n_sides, s + n_sides])
        idx_k.extend([ns + n_sides, s + n_sides, s])
    return verts, idx_i, idx_j, idx_k

# ============================================================
# SUBDIVISION LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            if a != b_: tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x,y in tri:
            if j==1 and not (1<=x<=y<=3): ok=False
            if j==2 and not (0<=x<=y-1 and 1<=y<=3): ok=False
            if j==3 and not (0<=x<=y<=2): ok=False
        if ok: return j
    return 3

tris = get_base_triangles()
parent_classes = [classify(t) for t in tris]

parent_simplices = []
for t in tris:
    simplex = np.vstack([np.array([[x,y,0] for x,y in t]), apex_main])
    parent_simplices.append(simplex)

def get_affine_params(P_from, P_to):
    A_target, A_orig = np.zeros((3,3)), np.zeros((3,3))
    for k in range(3):
        A_target[:,k] = P_to[k+1] - P_to[0]
        A_orig[:,k] = P_from[k+1] - P_from[0]
    M = A_target @ np.linalg.inv(A_orig)
    return M, P_to[0] - M @ P_from[0]

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

t_fixed = 1/4
frozen_children = []
for tri in tris:
    j = classify(tri)
    c = np.vstack([np.array([[x,y,0] for x,y in tri]), apex_main])
    shift = np.array([*(t_fixed*(b_vec_2d - 3*np.array(v_map[j]))), 0])
    frozen_children.append(c + shift)

# ============================================================
# ANIMATION & FIGURE ASSEMBLY
# ============================================================
p_fixed = 0.25
s_steps = np.linspace(0, 0.25, 20)

def create_frame_data(s):
    frame_traces = []
    wires_coords = {1: [[], [], []], 2: [[], [], []], 3: [[], [], []]}

    for i, (M, trans) in enumerate(parent_maps):
        j_p = parent_classes[i]
        bary_shift = p_fixed * (global_barycenter - np.mean(parent_simplices[i][:3], 0))
        block_shift = np.array([*(s * (b_vec_2d - 3 * np.array(v_map[j_p]))), 0])
        total_shift = bary_shift + block_shift

        for child in frozen_children:
            pos = (M @ child.T).T + trans + total_shift

            tube_data = get_rotated_tube(pos, radius=1/36, height_factor=0.75)
            if tube_data:
                tx, ti, tj, tk = tube_data
                frame_traces.append(go.Mesh3d(
                    x=tx[:,0], y=tx[:,1], z=tx[:,2],
                    i=ti, j=tj, k=tk,
                    color=wire_colors[j_p], opacity=1.0, flatshading=True, showlegend=False
                ))

            wx, wy, wz = get_simplex_skeleton(pos)
            wires_coords[j_p][0].extend(wx)
            wires_coords[j_p][1].extend(wy)
            wires_coords[j_p][2].extend(wz)

    for c_idx, coords in wires_coords.items():
        frame_traces.append(go.Scatter3d(
            x=coords[0], y=coords[1], z=coords[2],
            mode='lines', line=dict(color=wire_colors[c_idx], width=1.5), showlegend=False
        ))
    return frame_traces

frames = [go.Frame(data=create_frame_data(s), name=f"s={s:.2f}") for s in s_steps]
fig = go.Figure(data=frames[0].data, frames=frames)

# Removal of axes, background, and grid
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False, showbackground=False),
        yaxis=dict(visible=False, showbackground=False),
        zaxis=dict(visible=False, showbackground=False, range=[0, 1.1]),
        aspectmode="data",
        bgcolor="rgba(0,0,0,0)"
    ),
    paper_bgcolor='white',
    width=1000, height=800,
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 50, "redraw": True}}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}}], "label": "Pause", "method": "animate"}
        ],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}}],
                   "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
        "x": 0.1, "len": 0.9,
        "currentvalue": {"font": {"size": 20}, "prefix": "t_2-value: ", "visible": True, "xanchor": "center"},
    }]
)

fig.show()

Added the reflected pyramids and the inscribed tubes reflections

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1. GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
global_barycenter = np.array([1.0, 2.0, 0.0])
homothetic_factor = 0.5

v_map = {1: [1,1], 2: [0,1], 3: [0,0]}
wire_colors = {1: "darkorange", 2: "indigo", 3: "darkgreen"}

# ============================================================
# 2. HELPERS: WIREFRAME & ROTATED TUBE
# ============================================================
def get_simplex_skeleton(vtx):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]
    return xs, ys, zs

def get_rotated_tube(pts, radius=1/36, height_factor=0.75, n_sides=12):
    base_center = np.mean(pts[:3], axis=0)
    apex = pts[3]
    vec = apex - base_center
    length = np.linalg.norm(vec)
    if length < 1e-6: return None
    direction = vec / length

    h = length * height_factor
    theta = np.linspace(0, 2*np.pi, n_sides, endpoint=False)
    cx, cy = radius * np.cos(theta), radius * np.sin(theta)

    v_bot = np.column_stack([cx, cy, np.zeros(n_sides)])
    v_top = np.column_stack([cx, cy, np.full(n_sides, h)])
    verts = np.vstack([v_bot, v_top])

    z_axis = np.array([0, 0, 1])
    if np.allclose(direction, z_axis): R = np.eye(3)
    elif np.allclose(direction, -z_axis): R = np.diag([1, -1, -1])
    else:
        v = np.cross(z_axis, direction)
        s, c = np.linalg.norm(v), np.dot(z_axis, direction)
        v_skew = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + v_skew + (v_skew @ v_skew) * ((1 - c) / (s**2))

    verts = (R @ verts.T).T + base_center
    idx_i, idx_j, idx_k = [], [], []
    for s in range(n_sides):
        ns = (s + 1) % n_sides
        idx_i.extend([s, s, ns + n_sides])
        idx_j.extend([ns, ns + n_sides, s + n_sides])
        idx_k.extend([ns + n_sides, s + n_sides, s])
    return verts, idx_i, idx_j, idx_k

# ============================================================
# 3. SUBDIVISION LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            if a != b_: tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x,y in tri:
            if j==1 and not (1<=x<=y<=3): ok=False
            if j==2 and not (0<=x<=y-1 and 1<=y<=3): ok=False
            if j==3 and not (0<=x<=y<=2): ok=False
        if ok: return j
    return 3

tris = get_base_triangles()
parent_classes = [classify(t) for t in tris]

parent_simplices = []
for t in tris:
    simplex = np.vstack([np.array([[x,y,0] for x,y in t]), apex_main])
    parent_simplices.append(simplex)

def get_affine_params(P_from, P_to):
    A_target, A_orig = np.zeros((3,3)), np.zeros((3,3))
    for k in range(3):
        A_target[:,k] = P_to[k+1] - P_to[0]
        A_orig[:,k] = P_from[k+1] - P_from[0]
    M = A_target @ np.linalg.inv(A_orig)
    return M, P_to[0] - M @ P_from[0]

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

t_fixed = 1/4
frozen_children = []
for tri in tris:
    j = classify(tri)
    c = np.vstack([np.array([[x,y,0] for x,y in tri]), apex_main])
    shift = np.array([*(t_fixed*(b_vec_2d - 3*np.array(v_map[j]))), 0])
    frozen_children.append(c + shift)

# ============================================================
# 4. ANIMATION FRAME GENERATION
# ============================================================
p_fixed = 0.25
s_steps = np.linspace(0, 0.25, 20)

def create_frame_data(s):
    frame_traces = []
    wires_coords = {1: [[], [], []], 2: [[], [], []], 3: [[], [], []]}
    refl_wires_coords = {1: [[], [], []], 2: [[], [], []], 3: [[], [], []]}

    for i, (M, trans) in enumerate(parent_maps):
        j_p = parent_classes[i]
        bary_shift = p_fixed * (global_barycenter - np.mean(parent_simplices[i][:3], 0))
        block_shift = np.array([*(s * (b_vec_2d - 3 * np.array(v_map[j_p]))), 0])
        total_shift = bary_shift + block_shift

        for child in frozen_children:
            pos = (M @ child.T).T + trans + total_shift
            apex = pos[3]

            # Primary Tube
            tube_data = get_rotated_tube(pos, radius=1/36, height_factor=0.75)
            if tube_data:
                tx, ti, tj, tk = tube_data
                frame_traces.append(go.Mesh3d(
                    x=tx[:,0], y=tx[:,1], z=tx[:,2],
                    i=ti, j=tj, k=tk,
                    color=wire_colors[j_p], opacity=1.0, flatshading=True, showlegend=False
                ))
                # Reflected Tube
                tx_refl = 2 * apex - tx
                frame_traces.append(go.Mesh3d(
                    x=tx_refl[:,0], y=tx_refl[:,1], z=tx_refl[:,2],
                    i=ti, j=tj, k=tk,
                    color=wire_colors[j_p], opacity=0.7, flatshading=True, showlegend=False
                ))

            # Wireframes
            wx, wy, wz = get_simplex_skeleton(pos)
            wires_coords[j_p][0].extend(wx); wires_coords[j_p][1].extend(wy); wires_coords[j_p][2].extend(wz)

            pos_refl = 2 * apex - pos
            rwx, rwy, rwz = get_simplex_skeleton(pos_refl)
            refl_wires_coords[j_p][0].extend(rwx); refl_wires_coords[j_p][1].extend(rwy); refl_wires_coords[j_p][2].extend(rwz)

    for c_idx in [1, 2, 3]:
        frame_traces.append(go.Scatter3d(x=wires_coords[c_idx][0], y=wires_coords[c_idx][1], z=wires_coords[c_idx][2],
                                        mode='lines', line=dict(color=wire_colors[c_idx], width=2), showlegend=False))
        frame_traces.append(go.Scatter3d(x=refl_wires_coords[c_idx][0], y=refl_wires_coords[c_idx][1], z=refl_wires_coords[c_idx][2],
                                        mode='lines', line=dict(color=wire_colors[c_idx], width=1), opacity=0.5, showlegend=False))
    return frame_traces

# ============================================================
# 5. FIGURE ASSEMBLY & CLEAN LAYOUT
# ============================================================
frames = [go.Frame(data=create_frame_data(s), name=f"s={s:.2f}") for s in s_steps]
fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(

    paper_bgcolor='white',
    plot_bgcolor='white',
    scene=dict(
        xaxis=dict(visible=False, showbackground=False),
        yaxis=dict(visible=False, showbackground=False),
        zaxis=dict(visible=False, showbackground=False, range=[0, 2.2]),
        aspectmode="data",
        bgcolor="white"
    ),
    width=1000, height=800,
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 50, "redraw": True}}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}}], "label": "Pause", "method": "animate"}
        ],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}}],
                   "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
        "x": 0.1, "len": 0.9,
        "currentvalue": {"font": {"size": 16}, "prefix": "t_2: ", "visible": True, "xanchor": "center"},
    }]
)

fig.show()

## $\delta$-balls incribed inside of each pyramid in the Perron tree construction, centered at the midpoint between the base barycenter and the apex

In [ ]:
# @title
import numpy as np
import plotly.graph_objects as go

# ============================================================
# GEOMETRY & CONSTANTS
# ============================================================
P1 = np.array([[0,0,0], [0,3,0], [3,3,0], [1,2,1]], dtype=float)
apex_main = np.array([1,2,1])
b_vec_2d = np.array([1.0, 2.0])
global_barycenter = np.array([1.0, 2.0, 0.0])
homothetic_factor = 0.5

v_map = {1: [1,1], 2: [0,1], 3: [0,0]}
wire_colors = {1: "darkorange", 2: "indigo", 3: "darkgreen"}

# ============================================================
# HELPERS: WIREFRAME & SPHERE MESH
# ============================================================
def get_simplex_skeleton(vtx):
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [vtx[a,0], vtx[b,0], None]
        ys += [vtx[a,1], vtx[b,1], None]
        zs += [vtx[a,2], vtx[b,2], None]
    return xs, ys, zs

def get_sphere_mesh(center, radius=1/18, res=12):
    """Generates vertices and indices for a sphere."""
    phi = np.linspace(0, np.pi, res)
    theta = np.linspace(0, 2*np.pi, res)
    phi, theta = np.meshgrid(phi, theta)

    x = center[0] + radius * np.sin(phi) * np.cos(theta)
    y = center[1] + radius * np.sin(phi) * np.sin(theta)
    z = center[2] + radius * np.cos(phi)

    verts = np.vstack([x.flatten(), y.flatten(), z.flatten()]).T

    idx_i, idx_j, idx_k = [], [], []
    for i in range(res - 1):
        for j in range(res - 1):
            p1 = i * res + j
            p2 = i * res + (j + 1)
            p3 = (i + 1) * res + (j + 1)
            p4 = (i + 1) * res + j
            idx_i.extend([p1, p1])
            idx_j.extend([p2, p3])
            idx_k.extend([p3, p4])

    return verts, idx_i, idx_j, idx_k

# ============================================================
# SUBDIVISION LOGIC
# ============================================================
def get_base_triangles():
    tris = []
    for a in range(3):
        for b_ in range(a, 3):
            tris.append([(a, b_), (a, b_+1), (a+1, b_+1)])
            if a != b_: tris.append([(a, b_), (a+1, b_), (a+1, b_+1)])
    return tris

def classify(tri):
    for j in [1, 2, 3]:
        ok = True
        for x,y in tri:
            if j==1 and not (1<=x<=y<=3): ok=False
            if j==2 and not (0<=x<=y-1 and 1<=y<=3): ok=False
            if j==3 and not (0<=x<=y<=2): ok=False
        if ok: return j
    return 3

tris = get_base_triangles()
parent_classes = [classify(t) for t in tris]

parent_simplices = []
for t in tris:
    simplex = np.vstack([np.array([[x,y,0] for x,y in t]), apex_main])
    parent_simplices.append(simplex)

def get_affine_params(P_from, P_to):
    A_target, A_orig = np.zeros((3,3)), np.zeros((3,3))
    for k in range(3):
        A_target[:,k] = P_to[k+1] - P_to[0]
        A_orig[:,k] = P_from[k+1] - P_from[0]
    M = A_target @ np.linalg.inv(A_orig)
    return M, P_to[0] - M @ P_from[0]

parent_maps = [get_affine_params(P1, T) for T in parent_simplices]

t_fixed = 1/4
frozen_children = []
for tri in tris:
    j = classify(tri)
    c = np.vstack([np.array([[x,y,0] for x,y in tri]), apex_main])
    shift = np.array([*(t_fixed*(b_vec_2d - 3*np.array(v_map[j]))), 0])
    frozen_children.append(c + shift)

# ============================================================
# ANIMATION & FIGURE ASSEMBLY
# ============================================================
p_fixed = 0.25
s_steps = np.linspace(0, 0.25, 20)

def create_frame_data(s):
    frame_traces = []
    wires_coords = {1: [[], [], []], 2: [[], [], []], 3: [[], [], []]}

    for i, (M, trans) in enumerate(parent_maps):
        j_p = parent_classes[i]
        bary_shift = p_fixed * (global_barycenter - np.mean(parent_simplices[i][:3], 0))
        block_shift = np.array([*(s * (b_vec_2d - 3 * np.array(v_map[j_p]))), 0])
        total_shift = bary_shift + block_shift

        for child in frozen_children:
            pos = (M @ child.T).T + trans + total_shift

            # Midpoint logic: center of gravity of the base to the apex
            base_center = np.mean(pos[:3], axis=0)
            apex = pos[3]
            ball_center = 0.5 * (base_center + apex)

            # RADIUS UPDATED TO 1/18
            bx, bi, bj, bk = get_sphere_mesh(ball_center, radius=1/18)
            frame_traces.append(go.Mesh3d(
                x=bx[:,0], y=bx[:,1], z=bx[:,2],
                i=bi, j=bj, k=bk,
                color=wire_colors[j_p],
                opacity=1.0,
                flatshading=False,
                lighting=dict(ambient=0.4, diffuse=0.8, specular=1, roughness=0.3),
                showlegend=False
            ))

            wx, wy, wz = get_simplex_skeleton(pos)
            wires_coords[j_p][0].extend(wx)
            wires_coords[j_p][1].extend(wy)
            wires_coords[j_p][2].extend(wz)

    for c_idx, coords in wires_coords.items():
        frame_traces.append(go.Scatter3d(
            x=coords[0], y=coords[1], z=coords[2],
            mode='lines', line=dict(color=wire_colors[c_idx], width=1.5), showlegend=False
        ))
    return frame_traces

frames = [go.Frame(data=create_frame_data(s), name=f"s={s:.2f}") for s in s_steps]
fig = go.Figure(data=frames[0].data, frames=frames)

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False, range=[0, 1.1]),
        aspectmode="data",
        bgcolor="white"
    ),
    paper_bgcolor='white',
    width=1000, height=800,
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 50, "redraw": True}}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}}], "label": "Pause", "method": "animate"}
        ],
        "type": "buttons", "x": 0.1, "y": 0.05
    }],
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}}],
                   "label": f"{float(f.name.split('=')[1]):.2f}", "method": "animate"} for f in frames],
        "x": 0.1, "len": 0.9,
    }]
)

fig.show()

Inscribed $\delta$-tubes whose distance to the top apex is exactly 1

In [ ]:
# @title

import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1. GEOMETRY & CONSTANTS
# ============================================================
A = np.array([0, 0, 0])
B = np.array([0, 3, 0])
C = np.array([3, 3, 0])
apex = np.array([1, 2, 2])

n_subdivisions = 9
tube_radius = 1 / 36
tube_height = 1.0
dist_top_to_apex = 1.0

# ============================================================
# 2. HELPER: TUBE POSITIONED BY TOP FACE DISTANCE
# ============================================================
def get_top_distanced_tube(base_center, target_apex, radius, height, top_dist, n_sides=16):
    vec = target_apex - base_center
    direction = vec / np.linalg.norm(vec)

    # Top face is 'top_dist' from apex; bottom face is 'top_dist + height' from apex
    tube_bottom = target_apex - (direction * (top_dist + height))

    theta = np.linspace(0, 2*np.pi, n_sides, endpoint=False)
    cx, cy = radius * np.cos(theta), radius * np.sin(theta)

    v_bot = np.column_stack([cx, cy, np.zeros(n_sides)])
    v_top = np.column_stack([cx, cy, np.full(n_sides, height)])
    verts = np.vstack([v_bot, v_top])

    z_axis = np.array([0, 0, 1])
    if np.allclose(direction, z_axis): R = np.eye(3)
    elif np.allclose(direction, -z_axis): R = np.diag([1, -1, -1])
    else:
        v = np.cross(z_axis, direction)
        s, c = np.linalg.norm(v), np.dot(z_axis, direction)
        v_skew = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + v_skew + (v_skew @ v_skew) * ((1 - c) / (s**2))

    verts = (R @ verts.T).T + tube_bottom

    idx_i, idx_j, idx_k = [], [], []
    for s in range(n_sides):
        ns = (s + 1) % n_sides
        idx_i.extend([s, s, ns + n_sides])
        idx_j.extend([ns, ns + n_sides, s + n_sides])
        idx_k.extend([ns + n_sides, s + n_sides, s])
    return verts, idx_i, idx_j, idx_k

def get_simplex_skeleton(v1, v2, v3, v4):
    pts = np.vstack([v1, v2, v3, v4])
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [pts[a,0], pts[b,0], None]
        ys += [pts[a,1], pts[b,1], None]
        zs += [pts[a,2], pts[b,2], None]
    return xs, ys, zs

# ============================================================
# 3. ASSEMBLY
# ============================================================
fig = go.Figure()
wire_x, wire_y, wire_z = [], [], []

for i in range(n_subdivisions):
    for j in range(n_subdivisions - i):
        # Base vertices
        v1 = (i*A + j*B + (n_subdivisions-i-j)*C)/n_subdivisions
        v2 = ((i+1)*A + j*B + (n_subdivisions-i-1-j)*C)/n_subdivisions
        v3 = (i*A + (j+1)*B + (n_subdivisions-i-j-1)*C)/n_subdivisions

        # Upward Cell
        b_cntr = (v1 + v2 + v3) / 3
        tube = get_top_distanced_tube(b_cntr, apex, tube_radius, tube_height, dist_top_to_apex)
        if tube:
            verts, ii, jj, kk = tube
            fig.add_trace(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2], i=ii, j=jj, k=kk,
                                    color='darkorange', opacity=1.0, flatshading=True))

        wx, wy, wz = get_simplex_skeleton(v1, v2, v3, apex)
        wire_x.extend(wx); wire_y.extend(wy); wire_z.extend(wz)

        # Downward Cell
        if i + j + 1 < n_subdivisions:
            v1d = ((i+1)*A + j*B + (n_subdivisions-i-1-j)*C)/n_subdivisions
            v2d = (i*A + (j+1)*B + (n_subdivisions-i-j-1)*C)/n_subdivisions
            v3d = ((i+1)*A + (j+1)*B + (n_subdivisions-i-j-2)*C)/n_subdivisions

            b_cntrd = (v1d + v2d + v3d) / 3
            tube_d = get_top_distanced_tube(b_cntrd, apex, tube_radius, tube_height, dist_top_to_apex)
            if tube_d:
                verts, ii, jj, kk = tube_d
                fig.add_trace(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2], i=ii, j=jj, k=kk,
                                        color='indigo', opacity=1.0, flatshading=True))

            wx, wy, wz = get_simplex_skeleton(v1d, v2d, v3d, apex)
            wire_x.extend(wx); wire_y.extend(wy); wire_z.extend(wz)

# Increased wireframe thickness to 3
fig.add_trace(go.Scatter3d(
    x=wire_x, y=wire_y, z=wire_z,
    mode='lines',
    line=dict(color='black', width=3),
    opacity=0.12
))

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        bgcolor='white'
    ),
    width=1000, height=800, showlegend=False, margin=dict(l=0, r=0, b=0, t=0)
)
fig.show()

In [ ]:
# @title

import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1. GEOMETRY & CONSTANTS
# ============================================================
A = np.array([0, 0, 0])
B = np.array([0, 3, 0])
C_base = np.array([3, 3, 0])
apex = np.array([1, 2, 2])

n_subdivisions = 9
ball_radius = 1 / 18

# ============================================================
# 2. HELPERS: SPHERE MESH & WIREFRAME
# ============================================================
def get_sphere_mesh(center, radius, res=16):
    """Generates a sphere mesh centered at a specific point."""
    phi = np.linspace(0, np.pi, res)
    theta = np.linspace(0, 2 * np.pi, res)
    phi, theta = np.meshgrid(phi, theta)

    x = center[0] + radius * np.sin(phi) * np.cos(theta)
    y = center[1] + radius * np.sin(phi) * np.sin(theta)
    z = center[2] + radius * np.cos(phi)

    verts = np.vstack([x.flatten(), y.flatten(), z.flatten()]).T

    idx_i, idx_j, idx_k = [], [], []
    for i in range(res - 1):
        for j in range(res - 1):
            p1 = i * res + j
            p2 = i * res + (j + 1)
            p3 = (i + 1) * res + (j + 1)
            p4 = (i + 1) * res + j
            idx_i.extend([p1, p1])
            idx_j.extend([p2, p3])
            idx_k.extend([p3, p4])

    return verts, idx_i, idx_j, idx_k

def get_simplex_skeleton(v1, v2, v3, v4):
    pts = np.vstack([v1, v2, v3, v4])
    edges = [(0,1), (1,2), (2,0), (0,3), (1,3), (2,3)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [pts[a,0], pts[b,0], None]
        ys += [pts[a,1], pts[b,1], None]
        zs += [pts[a,2], pts[b,2], None]
    return xs, ys, zs

# ============================================================
# 3. ASSEMBLY
# ============================================================
fig = go.Figure()
wire_x, wire_y, wire_z = [], [], []

for i in range(n_subdivisions):
    for j in range(n_subdivisions - i):
        # ----------------------------------------------------
        # Upward-Pointing Cell
        # ----------------------------------------------------
        v1 = (i*A + j*B + (n_subdivisions-i-j)*C_base)/n_subdivisions
        v2 = ((i+1)*A + j*B + (n_subdivisions-i-1-j)*C_base)/n_subdivisions
        v3 = (i*A + (j+1)*B + (n_subdivisions-i-j-1)*C_base)/n_subdivisions

        # Calculate midpoint between base barycenter and apex
        base_center = (v1 + v2 + v3) / 3
        ball_center = 0.5 * (base_center + apex)

        # Add Sphere
        sv, si, sj, sk = get_sphere_mesh(ball_center, ball_radius)
        fig.add_trace(go.Mesh3d(x=sv[:,0], y=sv[:,1], z=sv[:,2], i=si, j=sj, k=sk,
                                color='darkorange', opacity=1.0))

        # Add Wireframe to list
        wx, wy, wz = get_simplex_skeleton(v1, v2, v3, apex)
        wire_x.extend(wx); wire_y.extend(wy); wire_z.extend(wz)

        # ----------------------------------------------------
        # Downward-Pointing Cell
        # ----------------------------------------------------
        if i + j + 1 < n_subdivisions:
            v1d = ((i+1)*A + j*B + (n_subdivisions-i-1-j)*C_base)/n_subdivisions
            v2d = (i*A + (j+1)*B + (n_subdivisions-i-j-1)*C_base)/n_subdivisions
            v3d = ((i+1)*A + (j+1)*B + (n_subdivisions-i-j-2)*C_base)/n_subdivisions

            base_center_d = (v1d + v2d + v3d) / 3
            ball_center_d = 0.5 * (base_center_d + apex)

            # Add Sphere
            sv, si, sj, sk = get_sphere_mesh(ball_center_d, ball_radius)
            fig.add_trace(go.Mesh3d(x=sv[:,0], y=sv[:,1], z=sv[:,2], i=si, j=sj, k=sk,
                                    color='indigo', opacity=1.0))

            # Add Wireframe to list
            wx, wy, wz = get_simplex_skeleton(v1d, v2d, v3d, apex)
            wire_x.extend(wx); wire_y.extend(wy); wire_z.extend(wz)

# Thicker wireframe (width=3)
fig.add_trace(go.Scatter3d(
    x=wire_x, y=wire_y, z=wire_z,
    mode='lines',
    line=dict(color='black', width=3),
    opacity=0.15
))

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        bgcolor='white'
    ),
    width=1000, height=800, showlegend=False, margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()

## Counterexample in $R_{4,4}$ to non intersection of cells if the minimal index choice is not followed

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np
from itertools import permutations

# --- 1. GEOMETRY AND UTILS ---
def is_in_R43(x):
    eps = 1e-9
    return (-eps <= x[0] <= x[1] + eps and x[1] <= x[2] + eps and x[2] <= 3 + eps)

def get_valid_indices(vertices):
    vs = {1: [1, 1, 1], 2: [0, 1, 1], 3: [0, 0, 1], 4: [0, 0, 0]}
    valid = []
    for i in range(1, 5):
        v_ref = np.array(vs[i])
        if all(is_in_R43(v - v_ref) for v in vertices):
            valid.append(i)
    return valid

# --- 2. GENERATE ALL CELLS ---
all_cells = []
for y1 in range(4):
    for y2 in range(y1, 4):
        for y3 in range(y2, 4):
            base = np.array([y1, y2, y3], dtype=float)
            for p in permutations([0, 1, 2]):
                verts = [base.copy()]
                curr = base.copy()
                for axis in p:
                    curr[axis] += 1
                    verts.append(curr.copy())
                v_np = np.array(verts)
                if all(0 <= v[0] <= v[1]+1e-7 and v[1] <= v[2]+1e-7 and v[2] <= 4+1e-7 for v in v_np):
                    all_cells.append({'verts': v_np, 'indices': get_valid_indices(v_np)})

# --- 3. ANIMATION LOGIC ---
def create_r44_minimal_view(mode='Arbitrary (Collision)'):
    t_values = np.linspace(0, 0.25, 20)

    idx_colors_muted = {1: '#ffcccc', 2: '#ccccff', 3: '#ccffcc', 4: '#ffffcc'}
    idx_colors_solid = {1: 'red', 2: 'blue', 3: 'green', 4: 'yellow'}
    idx_colors_dark = {1: '#8b0000', 2: '#00008b', 3: '#006400', 4: '#8b8b00'}

    target_pair = []
    for i, c in enumerate(all_cells):
        if 3 in c['indices'] and 4 in c['indices']:
            for j, other in enumerate(all_cells):
                if i != j and 3 in other['indices'] and 4 in other['indices']:
                    if c['verts'][:,2].mean() < other['verts'][:,2].mean():
                        target_pair = [i, j]
                        break
            if target_pair: break

    frames = []
    edge_idx = [0,1,2,0,3,1,3,2]

    for t in t_values:
        frame_data = []
        for i, cell in enumerate(all_cells):
            is_collision_cell = i in target_pair
            if mode == 'Minimal Index (Paper)':
                idx = min(cell['indices'])
            else:
                if i == target_pair[0]: idx = 3
                elif i == target_pair[1]: idx = 4
                else: idx = min(cell['indices'])

            disps = {
                1: np.array([3*t, 2*t, 1*t]),
                2: np.array([-1*t, 2*t, 1*t]),
                3: np.array([-1*t, -2*t, 1*t]),
                4: np.array([-1*t, -2*t, -3*t])
            }

            shifted = cell['verts'] + disps[idx]
            x, y, z = shifted[:, 0], shifted[:, 1], shifted[:, 2]

            frame_data.append(go.Mesh3d(
                x=x, y=y, z=z, i=[0,0,0,1], j=[1,1,2,2], k=[2,3,3,3],
                color=idx_colors_solid[idx] if is_collision_cell else idx_colors_muted[idx],
                opacity=1.0 if is_collision_cell else 0.1,
                flatshading=True, showlegend=False
            ))

            # UPDATED: line color now uses idx_colors_dark[idx] even for collision cells
            frame_data.append(go.Scatter3d(
                x=x[edge_idx], y=y[edge_idx], z=z[edge_idx],
                mode='lines',
                line=dict(
                    color=idx_colors_dark[idx],
                    width=6 if is_collision_cell else 1.5
                ),
                opacity=1.0 if is_collision_cell else 0.4,
                showlegend=False
            ))

        frames.append(go.Frame(data=frame_data, name=f"t={t:.2f}"))

    fig = go.Figure(data=frames[0].data, frames=frames)

    fig.update_layout(

        paper_bgcolor='white',
        plot_bgcolor='white',
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor='white',
            aspectmode='cube'
        ),
        updatemenus=[{"buttons": [{"args": [None, {"frame": {"duration": 150, "redraw": True}}], "label": "Play", "method": "animate"}],
                      "type": "buttons", "x": 0.1, "y": 0}],
        sliders=[{"steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                             "label": f"{t_val:.2f}", "method": "animate"} for t_val, f in zip(t_values, frames)]}]
    )
    return fig

create_r44_minimal_view(mode='Arbitrary (Collision)').show()

Projection onto the slanted face of the previous plot

In [ ]:
# @title
import plotly.graph_objects as go
import numpy as np
from itertools import permutations

# --- 1. GEOMETRY UTILS ---
def is_in_R43(x):
    eps = 1e-8
    return (-eps <= x[0] <= x[1] + eps and x[1] <= x[2] + eps and x[2] <= 3 + eps)

def get_valid_indices(vertices):
    vs = {1: [1, 1, 1], 2: [0, 1, 1], 3: [0, 0, 1], 4: [0, 0, 0]}
    valid = []
    for i in range(1, 5):
        v_ref = np.array(vs[i])
        if all(is_in_R43(v - v_ref) for v in vertices):
            valid.append(i)
    return valid

# --- 2. GENERATE CELLS ON THE SLANTED FACE x1 = x2 ---
face_cells = []
for y1 in range(4):
    for y2 in range(y1, 4):
        for y3 in range(y2, 4):
            base = np.array([y1, y2, y3], dtype=float)
            for p in permutations([0, 1, 2]):
                verts = [base.copy()]
                curr = base.copy()
                for axis in p:
                    curr[axis] += 1
                    verts.append(curr.copy())
                v_np = np.array(verts)

                on_face = [v for v in v_np if abs(v[0] - v[1]) < 1e-7]
                if len(on_face) >= 3:
                    indices = get_valid_indices(v_np)
                    if indices:
                        face_cells.append({
                            'face_pts': np.array(on_face),
                            'indices': indices
                        })

# --- 3. MANUAL SLIDER ANIMATION ---
def create_manual_2d_collision(mode='Arbitrary (Collision)'):
    t_values = np.linspace(0, 0.25, 25)
    idx_colors_muted = {1: '#ffcccc', 2: '#ccccff', 3: '#ccffcc', 4: '#ffffcc'}
    idx_colors_solid = {1: 'red', 2: 'blue', 3: 'green', 4: 'yellow'}
    idx_colors_dark = {1: '#8b0000', 2: '#00008b', 3: '#006400', 4: '#8b8b00'}

    target_pair = []
    for i, c in enumerate(face_cells):
        if 3 in c['indices'] and 4 in c['indices']:
            for j, other in enumerate(face_cells):
                if i != j and 3 in other['indices'] and 4 in other['indices']:
                    if c['face_pts'][:,2].mean() < other['face_pts'][:,2].mean():
                        target_pair = [i, j]
                        break
            if target_pair: break

    frames = []
    for t in t_values:
        frame_data = []
        disps = {
            1: np.array([2*t, 1*t]), 2: np.array([2*t, 1*t]),
            3: np.array([-2*t, 1*t]), 4: np.array([-2*t, -3*t])
        }

        for i, cell in enumerate(face_cells):
            is_col = (len(target_pair) == 2 and i in target_pair)

            if mode == 'Minimal Index (Paper)':
                idx = min(cell['indices'])
            else:
                if is_col:
                    idx = 3 if i == target_pair[0] else 4
                else:
                    idx = min(cell['indices'])

            pts_2d = cell['face_pts'][:, 1:] + disps[idx]

            center = pts_2d.mean(axis=0)
            angles = np.arctan2(pts_2d[:,1]-center[1], pts_2d[:,0]-center[0])
            sorted_pts = pts_2d[np.argsort(angles)]
            sorted_pts = np.vstack([sorted_pts, sorted_pts[0]])

            # CHANGED: line color now uses idx_colors_dark[idx] even for collisions
            frame_data.append(go.Scatter(
                x=sorted_pts[:,0], y=sorted_pts[:,1],
                fill="toself", mode='lines',
                line=dict(
                    color=idx_colors_dark[idx],
                    width=3 if is_col else 1
                ),
                fillcolor=idx_colors_solid[idx] if is_col else idx_colors_muted[idx],
                opacity=1.0 if is_col else 0.2,
                showlegend=False
            ))
        frames.append(go.Frame(data=frame_data, name=f"{t:.3f}"))

    fig = go.Figure(data=frames[0].data, frames=frames)

    sliders_dict = {
        "active": 0, "yanchor": "top", "xanchor": "left",
        "currentvalue": {"font": {"size": 20}, "prefix": "t = ", "visible": True, "xanchor": "right"},
        "transition": {"duration": 0}, "pad": {"b": 10, "t": 50}, "len": 0.9, "x": 0.1, "y": 0,
        "steps": []
    }

    for t, frame in zip(t_values, frames):
        sliders_dict["steps"].append({
            "args": [[frame.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
            "label": f"{t:.2f}", "method": "animate"
        })

    fig.update_layout(

        width=750, height=750,
        sliders=[sliders_dict],
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(range=[-0.5, 4.5], visible=False, showgrid=False, zeroline=False),
        yaxis=dict(range=[-0.5, 4.5], visible=False, showgrid=False, zeroline=False, scaleanchor="x", scaleratio=1)
    )
    return fig

create_manual_2d_collision().show()

In [ ]:
# @title
import numpy as np
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import itertools

# 1. Define the 3D Simplex Vertices
V_cart = np.array([[0,0,0], [0,0,3], [0,3,3], [3,3,3]])

# 2. Find Exact Vertices via Barycentric Constraints
vals = [0.0, 0.25, 1.0]
constraints = []
for i in range(4):
    for v in vals:
        constraints.append((i, v))

all_pts = []
for combo in itertools.combinations(constraints, 3):
    indices = [c[0] for c in combo]
    if len(set(indices)) < 3: continue
    A = np.zeros((4, 4)); b = np.zeros(4)
    A[0, :] = 1; b[0] = 1.0
    for i, (idx, val) in enumerate(combo):
        A[i+1, idx] = 1; b[i+1] = val
    try:
        pt = np.linalg.solve(A, b)
        if np.all(pt >= -1e-10) and np.all(pt <= 1 + 1e-10):
            all_pts.append(np.round(pt, 8))
    except np.linalg.LinAlgError: pass

all_pts = np.unique(all_pts, axis=0)

# 3. Logic for Regions
eps = 1e-7
def in_Sigma_prime(pt, i):
    if pt[i] < 0.25 - eps: return False
    for j in range(i):
        if pt[j] > 0.25 + eps: return False
    return True

def in_Sigma_full(pt, i):
    return pt[i] >= 0.25 - eps

colors = ['#FF0000', '#0070FF', '#00FF00', '#FFD700']
fig = go.Figure()

for i in range(4):
    color = colors[i]

    # --- PLOT SOLID SIGMA'_i ---
    pts_prime_bary = np.array([pt for pt in all_pts if in_Sigma_prime(pt, i)])

    if len(pts_prime_bary) >= 4:
        pts_p_cart = pts_prime_bary @ V_cart
        if np.linalg.matrix_rank(pts_p_cart - np.mean(pts_p_cart, axis=0), tol=1e-8) >= 2:
            try:
                h_p = ConvexHull(pts_p_cart)
                fig.add_trace(go.Mesh3d(
                    x=pts_p_cart[:, 0], y=pts_p_cart[:, 1], z=pts_p_cart[:, 2],
                    i=h_p.simplices[:, 0], j=h_p.simplices[:, 1], k=h_p.simplices[:, 2],
                    color=color,
                    opacity=0.4, # Increased transparency
                    flatshading=True
                ))
            except:
                pass

    # --- PLOT WIREFRAME SIGMA_i ---
    pts_full_bary = np.array([pt for pt in all_pts if in_Sigma_full(pt, i)])
    if len(pts_full_bary) >= 4:
        pts_f_cart = pts_full_bary @ V_cart
        h_f = ConvexHull(pts_f_cart)

        edges = set()
        for simplex in h_f.simplices:
            for s1, s2 in itertools.combinations(simplex, 2):
                edges.add(tuple(sorted((s1, s2))))

        ex, ey, ez = [], [], []
        for edge in edges:
            p1, p2 = pts_f_cart[edge[0]], pts_f_cart[edge[1]]
            ex.extend([p1[0], p2[0], None])
            ey.extend([p1[1], p2[1], None])
            ez.extend([p1[2], p2[2], None])

        fig.add_trace(go.Scatter3d(
            x=ex, y=ey, z=ez, mode='lines',
            line=dict(color=color, width=6), # Thicker wireframe
            opacity=0.6, hoverinfo='skip'
        ))

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='white',
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.8))
    ),
    paper_bgcolor='white',
    margin=dict(l=0, r=0, b=0, t=0),
    showlegend=False,
    width=900, height=800
)

fig.show()